# ARC Prize 2026 — Online World-Model Induction & Hybrid RL-Search for ARC-AGI-3

Companion code release for the ARC Prize 2026 Paper Track writeup.
Two complementary agents for ARC-AGI-3, plus the local evaluation harness:

* **OCEAN** (`agent_core.py`) — training-free online world-model induction:
  action-semantics probing via pixel-diff motion, discrete-vs-slide mechanism
  classification, edge-blocking wall model, goal/hazard colour learning,
  frontier-BFS exploration, click/select fallbacks, and effective-object
  click dwelling for multi-press mechanics.
* **Forge** (`forge_agent.py`) — hybrid RL + symbolic search:
  ForgeNet (CBAM attention over a frame stack), prioritized experience replay,
  persistent cross-level archive, a background multi-solver cascade
  (Dijkstra transfer, beam search, IDA*, bounded BFS, A*, MCTS), and
  test-time training under a global time budget. **This is the agent in
  competition submission ref 56378196** (`takumuhata/forge-pathfinder-bfs-agent`).
* `run_local.py` — headless evaluator for the 25 local development games,
  reporting per-level action counts and RHAE scores.

## Results (25 dev games, ≤2,500 actions each)

| Agent | Games with ≥1 level | Levels | Mean game score |
|---|---|---|---|
| Random | 1/25 | 1 | ~0% |
| OCEAN | 11/25 | 9 | ~6.3% |
| Oracle solver (privileged dev tool) | 10/25 | 12 | — |
| Forge (submitted, ref 56378196) | — | — | 0.08 public LB |

Both agents run fully offline: no internet, no external model calls.

## Reproduce locally

```bash
pip install arc-agi                 # ARC-AGI-3 client library
git clone https://github.com/... ARC-AGI-3-Agents   # official framework
python run_local.py all 2500        # evaluate OCEAN on all dev games
python play_local.py --game m0r0    # drive any Agent subclass locally
```


In [ ]:
%%writefile agent_core.py
"""OCEAN: Object-Centric Exploration and Navigation agent for ARC-AGI-3.

A training-free, online world-model-induction agent. Per step it:
  * diffs consecutive frames to find which objects moved in response to the
    last action (robust in fields of identical tiles),
  * probes the four directional actions to identify the controllable object
    (the "avatar") by canonical-direction agreement, grouping co-moving sprite
    colors,
  * learns per-action motion (discrete step vs. slide-until-blocked), wall /
    hazard colors from failed or fatal moves, and the goal color from the
    completing action's destination,
  * plans with BFS over lattice positions toward hypothesized goal objects,
    falling back to systematic frontier exploration and object clicking.
  * GAME_OVER -> RESET; knowledge (walls, goal/click colors) persists across
    level-reset retries.
"""
from __future__ import annotations

import random
from collections import Counter, defaultdict, deque

import numpy as np

A_RESET = 0
SIMPLE = [1, 2, 3, 4, 5, 7]
CLICK = 6


def diff_mask(prev: np.ndarray, cur: np.ndarray) -> np.ndarray:
    return prev != cur


def color_motion(prev, cur, mask):
    """For each color: cells erased (prev==c & diff) and drawn (cur==c & diff).
    Returns {color: (dx, dy, n)} using centroid difference when both exist."""
    out = {}
    if not mask.any():
        return out
    ys, xs = np.where(mask)
    for c in np.unique(np.concatenate([prev[ys, xs], cur[ys, xs]])):
        old = mask & (prev == c)
        new = mask & (cur == c)
        no, nn = int(old.sum()), int(new.sum())
        if no and nn:
            oys, oxs = np.where(old)
            nys, nxs = np.where(new)
            out[int(c)] = (float(nxs.mean() - oxs.mean()),
                           float(nys.mean() - oys.mean()), min(no, nn))
        else:
            out[int(c)] = (0.0, 0.0, max(no, nn))
    return out


def components_of_color(grid, color, max_keep=200):
    """Connected components (4-conn) of a single color -> list of (cells_set, centroid)."""
    mask = grid == color
    h, w = grid.shape
    seen = np.zeros_like(mask, dtype=bool)
    comps = []
    ys, xs = np.where(mask & ~seen)
    for y0, x0 in zip(ys, xs):
        if seen[y0, x0] or not mask[y0, x0]:
            continue
        cells = []
        stack = [(int(x0), int(y0))]
        seen[y0, x0] = True
        while stack:
            x, y = stack.pop()
            cells.append((x, y))
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    stack.append((nx, ny))
        comps.append((cells, (np.mean([p[0] for p in cells]),
                              np.mean([p[1] for p in cells]))))
        if len(comps) >= max_keep:
            break
    return comps


def all_objects(grid):
    """Connected components per non-dominant color -> list of dicts."""
    h, w = grid.shape
    vals, counts = np.unique(grid, return_counts=True)
    bg = {int(v) for v, c in zip(vals, counts) if c > 0.45 * h * w}
    objs = []
    for v in vals:
        c = int(v)
        if c in bg:
            continue
        for cells, cent in components_of_color(grid, c):
            xs = [p[0] for p in cells]; ys = [p[1] for p in cells]
            objs.append({"color": c, "cells": cells, "centroid": cent,
                         "bbox": (min(xs), min(ys), max(xs), max(ys)),
                         "size": len(cells)})
    return objs


"""OCEAN: Object-Centric Exploration and Navigation agent for ARC-AGI-3.

A training-free, online world-model-induction agent. Per step it:
  * diffs consecutive frames to find which objects moved in response to the
    last action (robust in fields of identical tiles),
  * probes the four directional actions to identify the controllable object
    (the "avatar") by canonical-direction agreement, grouping co-moving sprite
    colors,
  * learns per-action motion (discrete step vs. slide-until-blocked), wall /
    hazard colors from failed or fatal moves, and the goal color from the
    completing action's destination,
  * plans with BFS over lattice positions toward hypothesized goal objects,
    falling back to systematic frontier exploration and object clicking.
  * GAME_OVER -> RESET; knowledge (walls, goal/click colors) persists across
    level-reset retries.
"""
from __future__ import annotations

import random
from collections import Counter, defaultdict, deque

import numpy as np

A_RESET = 0
SIMPLE = [1, 2, 3, 4, 5, 7]
CLICK = 6


def diff_mask(prev: np.ndarray, cur: np.ndarray) -> np.ndarray:
    return prev != cur


def color_motion(prev, cur, mask):
    """For each color: cells erased (prev==c & diff) and drawn (cur==c & diff).
    Returns {color: (dx, dy, n)} using centroid difference when both exist."""
    out = {}
    if not mask.any():
        return out
    ys, xs = np.where(mask)
    for c in np.unique(np.concatenate([prev[ys, xs], cur[ys, xs]])):
        old = mask & (prev == c)
        new = mask & (cur == c)
        no, nn = int(old.sum()), int(new.sum())
        if no and nn:
            oys, oxs = np.where(old)
            nys, nxs = np.where(new)
            out[int(c)] = (float(nxs.mean() - oxs.mean()),
                           float(nys.mean() - oys.mean()), min(no, nn))
        else:
            out[int(c)] = (0.0, 0.0, max(no, nn))
    return out


def components_of_color(grid, color, max_keep=200):
    """Connected components (4-conn) of a single color -> list of (cells_set, centroid)."""
    mask = grid == color
    h, w = grid.shape
    seen = np.zeros_like(mask, dtype=bool)
    comps = []
    ys, xs = np.where(mask & ~seen)
    for y0, x0 in zip(ys, xs):
        if seen[y0, x0] or not mask[y0, x0]:
            continue
        cells = []
        stack = [(int(x0), int(y0))]
        seen[y0, x0] = True
        while stack:
            x, y = stack.pop()
            cells.append((x, y))
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    stack.append((nx, ny))
        comps.append((cells, (np.mean([p[0] for p in cells]),
                              np.mean([p[1] for p in cells]))))
        if len(comps) >= max_keep:
            break
    return comps


def all_objects(grid):
    """Connected components per non-dominant color -> list of dicts."""
    h, w = grid.shape
    vals, counts = np.unique(grid, return_counts=True)
    bg = {int(v) for v, c in zip(vals, counts) if c > 0.45 * h * w}
    objs = []
    for v in vals:
        c = int(v)
        if c in bg:
            continue
        for cells, cent in components_of_color(grid, c):
            xs = [p[0] for p in cells]; ys = [p[1] for p in cells]
            objs.append({"color": c, "cells": cells, "centroid": cent,
                         "bbox": (min(xs), min(ys), max(xs), max(ys)),
                         "size": len(cells)})
    return objs


"""OCEAN: Object-Centric Exploration and Navigation agent for ARC-AGI-3.

A training-free, online world-model-induction agent. Per step it:
  * diffs consecutive frames to find which objects moved in response to the
    last action (robust in fields of identical tiles),
  * probes the four directional actions to identify the controllable object
    (the "avatar") by canonical-direction agreement, grouping co-moving sprite
    colors,
  * learns per-action motion (discrete step vs. slide-until-blocked), wall /
    hazard colors from failed or fatal moves, and the goal color from the
    completing action's destination,
  * plans with BFS over lattice positions toward hypothesized goal objects,
    falling back to systematic frontier exploration and object clicking.
  * GAME_OVER -> RESET; knowledge (walls, goal/click colors) persists across
    level-reset retries.
"""
from __future__ import annotations

import random
from collections import Counter, defaultdict, deque

import numpy as np

A_RESET = 0
SIMPLE = [1, 2, 3, 4, 5, 7]
CLICK = 6


def diff_mask(prev: np.ndarray, cur: np.ndarray) -> np.ndarray:
    return prev != cur


def color_motion(prev, cur, mask):
    """For each color: cells erased (prev==c & diff) and drawn (cur==c & diff).
    Returns {color: (dx, dy, n)} using centroid difference when both exist."""
    out = {}
    if not mask.any():
        return out
    ys, xs = np.where(mask)
    for c in np.unique(np.concatenate([prev[ys, xs], cur[ys, xs]])):
        old = mask & (prev == c)
        new = mask & (cur == c)
        no, nn = int(old.sum()), int(new.sum())
        if no and nn:
            oys, oxs = np.where(old)
            nys, nxs = np.where(new)
            out[int(c)] = (float(nxs.mean() - oxs.mean()),
                           float(nys.mean() - oys.mean()), min(no, nn))
        else:
            out[int(c)] = (0.0, 0.0, max(no, nn))
    return out


def components_of_color(grid, color, max_keep=200):
    """Connected components (4-conn) of a single color -> list of (cells_set, centroid)."""
    mask = grid == color
    h, w = grid.shape
    seen = np.zeros_like(mask, dtype=bool)
    comps = []
    ys, xs = np.where(mask & ~seen)
    for y0, x0 in zip(ys, xs):
        if seen[y0, x0] or not mask[y0, x0]:
            continue
        cells = []
        stack = [(int(x0), int(y0))]
        seen[y0, x0] = True
        while stack:
            x, y = stack.pop()
            cells.append((x, y))
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    stack.append((nx, ny))
        comps.append((cells, (np.mean([p[0] for p in cells]),
                              np.mean([p[1] for p in cells]))))
        if len(comps) >= max_keep:
            break
    return comps


def all_objects(grid):
    """Connected components per non-dominant color -> list of dicts."""
    h, w = grid.shape
    vals, counts = np.unique(grid, return_counts=True)
    bg = {int(v) for v, c in zip(vals, counts) if c > 0.45 * h * w}
    objs = []
    for v in vals:
        c = int(v)
        if c in bg:
            continue
        for cells, cent in components_of_color(grid, c):
            xs = [p[0] for p in cells]; ys = [p[1] for p in cells]
            objs.append({"color": c, "cells": cells, "centroid": cent,
                         "bbox": (min(xs), min(ys), max(xs), max(ys)),
                         "size": len(cells)})
    return objs


class OceanCore:
    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)
        # persistent across levels of the same game
        self.avatar_colors = set()
        self.dir = {}                  # action -> unit direction (sx,sy)
        self.mag = {}                  # action -> fixed step px (discrete moves)
        self.slide = {}                # action -> True if variable-distance move
        self.obs = defaultdict(list)   # action -> observed (dx,dy) displacements
        self.move_actions = []
        self.blocking_colors = set()   # colors that block avatar motion
        self.goal_color = None         # learned after first level completion
        self.click_color = None        # color of successfully-clicked objects
        self.hazard_count = defaultdict(int)  # color -> deaths at destination
        self._goal_locs = []           # positions where past levels completed
        self._collect_colors = set()   # colors of objects consumed pre-completion
        self.quantum = 1               # lattice step estimate (px)
        self.level_init()

    def level_init(self):
        self.grid = None
        self.phase = "probe"
        self.probe_queue = deque()
        self.probe_prev = None
        self.probe_action = None
        self.probe_results = []        # list of (action, color_motion dict)
        self.avatar_pos = None         # centroid (x,y) float
        self.avatar_cells = set()
        self.visited = set()           # avatar lattice positions visited
        self.blocked = set()           # lattice positions learned blocked
        self.bad_targets = set()
        self._target_obj = {}
        self.plan = deque()
        self._frozen = 0               # steps with tracked object unmoved
        self._stuck = 0                # consecutive failed moves at one cell
        self._traps = set()            # cells that immobilize the avatar
        self._objs_prev = None         # (color,x,y) objects seen last frame
        self._consumed = deque(maxlen=400)  # (color, level_actions) vanished
        self._safe_colors = set()      # colors the avatar has stood beside
        self._last = None              # (action, pos_before) for verification
        self.clicked = set()
        self._eff_obj = deque()        # effective objects: {"key", "cells", "presses"}
        self._eff_obj_keys = set()
        self._eff_turn = False         # alternate eff-dwelling with sweeping
        self._press_count = defaultdict(int)  # clicks issued per cell
        self._sweep = None             # ordered click-sweep positions
        self._click_burst = 0          # forced clicks while nav is stalled
        self.stuck = 0
        self.level_actions = 0
        self.cur_level = -1
        self.no_progress_steps = 0
        self._wcells = set()
        self.noop_streak = 0         # consecutive frames with zero change
        self.unlock_left = 0         # steps spent trying selection/clicks
        self._hist = deque(maxlen=8)  # recent diff masks
        self._ticking = None          # cells that change every frame (UI)

    # ================= main =================
    def decide(self, grid, available, state, levels_completed, step):
        if state in ("NOT_PLAYED", "GAME_OVER"):
            if state == "GAME_OVER":
                self._record_death()
            return (A_RESET, {})

        grid = np.asarray(grid)
        if self.grid is None:
            self.grid = grid
            self.cur_level = levels_completed
            self._build_probe(available)
            return self._consume_probe(grid, available)

        if levels_completed != self.cur_level:
            self._on_level_change(levels_completed, grid, available)
            return self._consume_probe(grid, available) if self.phase == "probe" \
                else self._act(grid, available)

        return self._act(grid, available) if self.phase != "probe" \
            else self._consume_probe(grid, available)

    def _on_level_change(self, lvl, grid, available):
        # The completing action's predicted destination was the goal —
        # learn its color so future levels can target it directly.
        if self._last:
            a, before = self._last[0], self._last[1]
            data = self._last[2] if len(self._last) > 2 else {}
            if a in self.dir and before is not None and self.avatar_colors:
                # the goal is the cell the avatar was entering — use its
                # actual final position + one step forward (predicted dest
                # overshoots on slides and misses the exit cell entirely)
                if self.avatar_pos is not None:
                    dest = (self.avatar_pos[0] + self.dir[a][0] * self.quantum,
                            self.avatar_pos[1] + self.dir[a][1] * self.quantum)
                else:
                    dest = self._predict_dest(before, a)
                gc = self._color_at(dest)
                if gc is not None and gc not in self.avatar_colors:
                    self.goal_color = gc
            elif a == CLICK and data.get("x") is not None:
                gc = self._color_at((data["x"], data["y"]))
                if gc is not None:
                    self.click_color = gc
        # remember where this level was completed — exits tend to sit in
        # consistent regions (e.g. the same edge), which biases later levels'
        # frontier choice even when no goal color is learnable
        if self.avatar_pos is not None:
            self._goal_locs.append(self._lattice(self.avatar_pos))
        # objects consumed shortly before completion are goal-relevant
        # (collect-them-all semantics) — prioritize their colors next level
        cnt = Counter(c for c, t in self._consumed
                      if self.level_actions - t < 300)
        for c, n in cnt.items():
            if n <= 10:
                # real pickups are consumed once; animated decorations
                # flicker in and out constantly — ignore those colors
                self._collect_colors.add(c)
        keep = (dict(self.dir), dict(self.mag), dict(self.slide),
                defaultdict(list, {a: list(v) for a, v in self.obs.items()}))
        keep_avatar = set(self.avatar_colors)
        keep_blocking = set(self.blocking_colors)
        self.level_init()
        self.cur_level = lvl   # after level_init, which resets it to -1
        self.dir, self.mag, self.slide, self.obs = keep
        self.avatar_colors = keep_avatar
        self.blocking_colors = keep_blocking
        self.move_actions = sorted(self.dir)
        self.grid = grid
        self.phase = "act" if self.avatar_colors else "probe"
        if self.phase == "probe":
            self._build_probe(available)
        else:
            self._locate_avatar(grid)

    # ================= probe =================
    def _build_probe(self, available):
        self.probe_queue = deque()
        self.probe_results = []
        # probe directions twice each; never probe ACTION5 (selection
        # switching corrupts tracking) or ACTION7 (undo reverts progress)
        simple = [a for a in available if a in (1, 2, 3, 4)]
        for a in simple + simple:
            self.probe_queue.append(a)
        self._probe_action = None
        self.probe_prev = None
        self.phase = "probe"

    def _consume_probe(self, grid, available):
        if self.probe_action is not None:
            mask = diff_mask(self.probe_prev, grid)
            self.probe_results.append((self.probe_action,
                                       color_motion(self.probe_prev, grid, mask)))
            self.probe_prev = grid
        if self.probe_queue:
            a = self.probe_queue.popleft()
            self.probe_action = a
            self.probe_prev = grid
            return (a, {})
        self.probe_action = None
        self._learn_semantics()
        self.phase = "act"
        self._locate_avatar(grid)
        return self._act(grid, available, first=True)

    # canonical directions per the ARC-AGI-3 docs (A1 up, A2 down, A3 left, A4 right)
    CANON = {1: (0, -1), 2: (0, 1), 3: (-1, 0), 4: (1, 0)}

    def _learn_semantics(self):
        # per color, per action -> list of (dx,dy) observations across passes
        per = defaultdict(lambda: defaultdict(list))
        for a, motion in self.probe_results:
            for c, (dx, dy, n) in motion.items():
                if abs(dx) > 0.5 or abs(dy) > 0.5:
                    per[c][a].append((dx, dy))
        # score each color by agreement with canonical action directions
        best_c, best_score = None, 1
        for c, amap in per.items():
            score = 0
            for a, lst in amap.items():
                if a not in self.CANON:
                    continue
                mdx = sorted(d[0] for d in lst)[len(lst)//2]
                mdy = sorted(d[1] for d in lst)[len(lst)//2]
                cx, cy = self.CANON[a]
                if mdx * cx > 0.5 or mdy * cy > 0.5:
                    score += 1
            if score > best_score:
                best_c, best_score = c, score
        if best_c is not None:
            # median delta per (color, action)
            med = {c: {a: (sorted(d[0] for d in lst)[len(lst)//2],
                           sorted(d[1] for d in lst)[len(lst)//2])
                       for a, lst in amap.items() if lst}
                   for c, amap in per.items()}
            # co-moving colors: same direction (within 2px) on >=1 shared action
            co = {best_c}
            for c in per:
                if c == best_c:
                    continue
                shared = sum(
                    1 for a in med[c]
                    if a in med[best_c]
                    and abs(med[c][a][0] - med[best_c][a][0]) <= 2
                    and abs(med[c][a][1] - med[best_c][a][1]) <= 2)
                if shared >= 1:
                    co.add(c)
            # per action: displacement obs from the co-moving color with most obs
            for a in range(1, 8):
                best_l, best_n = None, -1
                for c in co:
                    if a in per[c] and len(per[c][a]) > best_n:
                        best_l, best_n = per[c][a], len(per[c][a])
                if best_l:
                    self.obs[a].extend(
                        (int(round(x)), int(round(y))) for x, y in best_l)
            self.avatar_colors = co
            self._reclassify()
        else:
            self.avatar_colors = set()
            self.dir, self.mag, self.slide = {}, {}, {}
            self.move_actions = []

    def _reclassify(self):
        """Turn raw displacement observations into dir/mag/slide per action.
        Uses the modal (most common) displacement; ACTION7 is never a move."""
        for a, lst in self.obs.items():
            if not lst or a not in (1, 2, 3, 4):
                continue
            mdx, mdy = Counter(lst).most_common(1)[0][0]
            if abs(mdx) >= abs(mdy):
                self.dir[a] = (1 if mdx > 0 else -1, 0)
            else:
                self.dir[a] = (0, 1 if mdy > 0 else -1)
            mags = [max(abs(x), abs(y)) for x, y in lst]
            mc = Counter(mags)
            mode_mag, mode_n = mc.most_common(1)[0]
            if mode_n * 3 >= len(mags) * 2:
                # one dominant step size -> discrete move
                self.slide[a] = False
                self.mag[a] = max(1, mode_mag)
            else:
                self.slide[a] = True
        allm = [max(abs(x), abs(y)) for lst in self.obs.values() for x, y in lst
                if (x, y) != (0, 0)]
        if allm:
            self.quantum = max(1, Counter(allm).most_common(1)[0][0])
        self.move_actions = sorted(a for a in self.dir if a != 7)

    # ================= perception =================
    def _locate_avatar(self, grid, anchor=None, prev_grid=None, ray=None):
        """Avatar = the same-color object that *moved* most (most cells in the
        changed region vs prev frame), nearest the anchor. This tracks the
        controllable object even if its color or identity changes (selection
        games). Falls back to nearest avatar-colored component. `ray` is a
        (origin, unit_dir) corridor for slide moves — the avatar may land
        anywhere along it."""
        if not self.avatar_colors:
            return None
        ref = anchor if anchor is not None else self.avatar_pos

        def cdist(cx, cy):
            if ray is not None:
                (bx, by), (ex, ey) = ray
                proj = (cx - bx) * ex + (cy - by) * ey
                perp = abs((cx - bx) * ey - (cy - by) * ex)
                return perp + max(0.0, -proj - self.quantum)
            if ref is None:
                return 0
            return abs(cx - ref[0]) + abs(cy - ref[1])
        changed = None
        if prev_grid is not None and prev_grid.shape == grid.shape:
            m = (prev_grid != grid)
            if self._ticking is not None:
                m = m & ~self._ticking
            changed = set(zip(*np.where(m)[::-1]))
        mask = np.isin(grid, list(self.avatar_colors))
        h, w = mask.shape
        seen = np.zeros_like(mask, dtype=bool)
        comps = []
        ys, xs = np.where(mask)
        for y0, x0 in zip(ys, xs):
            if seen[y0, x0]:
                continue
            cells = []
            stack = [(int(x0), int(y0))]
            seen[y0, x0] = True
            while stack:
                x, y = stack.pop()
                cells.append((x, y))
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] and not seen[ny, nx]:
                        seen[ny, nx] = True
                        stack.append((nx, ny))
            comps.append(cells)
        if not comps:
            return None
        # continuity gate: when a reference (predicted or last-known pos)
        # exists, ignore far-away components — flickering UI elements that
        # share avatar colors otherwise hijack tracking (ar25's right-edge
        # phantom). Fall back to the full pool if nothing is near.
        if ref is not None or ray is not None:
            near = [c for c in comps
                    if cdist(sum(p[0] for p in c) / len(c),
                             sum(p[1] for p in c) / len(c))
                    <= 6 * self.quantum]
            if near:
                comps = near
        # rank components: most cells in the changed region (i.e. the object
        # that moved), then nearest to the reference point
        best, bkey = None, None
        for cells in comps:
            cx = sum(p[0] for p in cells) / len(cells)
            cy = sum(p[1] for p in cells) / len(cells)
            n_new = sum(1 for c in cells if c in changed) if changed else 0
            d = cdist(cx, cy)
            key = (-n_new, d)
            if bkey is None or key < bkey:
                best, bkey = (cells, (cx, cy)), key
        self.avatar_cells, self.avatar_pos = best
        xs2 = [p[0] for p in self.avatar_cells]
        ys2 = [p[1] for p in self.avatar_cells]
        self._av_dims = (max(xs2) - min(xs2) + 1, max(ys2) - min(ys2) + 1)
        return self.avatar_pos

    # ================= act =================
    def _act(self, grid, available, first=False):
        prev_grid = self.grid
        self.grid = grid
        self.level_actions += 1

        diff = None
        if prev_grid is not None and prev_grid.shape == grid.shape:
            diff = (prev_grid != grid)
            self._hist.append(diff)
            if len(self._hist) >= 4:
                # cells that changed in EVERY recent frame are UI (step
                # counters, animations) — exclude from "real" change
                self._ticking = np.logical_and.reduce(list(self._hist))

        # object consumption: objects present last frame but gone now were
        # collected/consumed — their colors identify goal-relevant pickups
        if prev_grid is not None and prev_grid.shape == grid.shape and \
                self.avatar_colors:
            prev_sig = self._objs_prev
            cur = {(o["color"], int(round(o["centroid"][0])),
                    int(round(o["centroid"][1])))
                   for o in all_objects(grid)
                   if o["color"] not in self.avatar_colors}
            if prev_sig is not None:
                for c, x, y in prev_sig:
                    if c in self.avatar_colors:
                        continue
                    if not any(c == c2 and abs(x - x2) <= 3 and
                               abs(y - y2) <= 3 for c2, x2, y2 in cur):
                        self._consumed.append((c, self.level_actions))
                        # a consumed object was touchable, not a wall — a
                        # failed bump may have wrongly learned it blocking
                        self.blocking_colors.discard(c)
            self._objs_prev = cur
        elif self.avatar_colors:
            self._objs_prev = {(o["color"], int(round(o["centroid"][0])),
                                int(round(o["centroid"][1])))
                               for o in all_objects(grid)
                               if o["color"] not in self.avatar_colors}

        if diff is None:
            frame_changed = True
        elif self._ticking is not None:
            frame_changed = bool((diff & ~self._ticking).any())
        else:
            frame_changed = bool(diff.any())
        if frame_changed:
            self.noop_streak = 0
        else:
            self.noop_streak += 1

        # multi-press / toggle click mechanics: a click that produced a
        # *substantial* change (≥8 non-ticking cells — animations and
        # counters are masked out) marks the cell "effective". Effective
        # cells are cycled round-robin, alternating with the sweep, which
        # covers both multi-press (lp85: same cell x5) and balance/toggle
        # (s5i5: alternate two cells) mechanics.
        if self._last and self._last[0] == CLICK:
            d = self._last[2] if len(self._last) > 2 else {}
            pos = (d.get("x"), d.get("y"))
            big_change = diff is not None and \
                int((diff & ~self._ticking).sum() if self._ticking is not None
                    else diff.sum()) >= 8
            if big_change and pos[0] is not None:
                # group by containing object — presses on the same object
                # accumulate (lp85: press a button 5x), while pressing a
                # different object can reset progress
                key, cells = self._obj_key(prev_grid, pos)
                if key is None:
                    key, cells = ("cell", pos), [pos]
                if key not in self._eff_obj_keys:
                    self._eff_obj_keys.add(key)
                    self._eff_obj.append({"key": key, "cells": cells,
                                          "presses": 0})

        if self.avatar_colors:
            anchor = None
            ray = None
            if self._last and self._last[0] in self.dir \
                    and self._last[1] is not None:
                if self.slide.get(self._last[0]):
                    # a slide can land anywhere along the ray — gate on a
                    # corridor, not the predicted endpoint
                    ray = (self._last[1], self.dir[self._last[0]])
                else:
                    anchor = self._predict_dest(self._last[1],
                                                self._last[0])
            if anchor is None and ray is None:
                anchor = self.avatar_pos  # continuity: avatar rarely jumps
            new_pos = self._locate_avatar(grid, anchor, prev_grid, ray=ray)
            self._verify_move(new_pos)
            if new_pos is None:
                # avatar gone (died?) -> reset handled by state; else re-scan
                self.avatar_pos = None
                self._locate_avatar(grid)
            # frozen-tracker recovery: if the tracked object has barely moved
            # all level, it is a static decoration sharing the palette
            # (level transitions can change the real avatar's colors).
            # Re-derive the palette from whatever actually changed.
            elif self._frozen >= 40 and len(self.visited) <= 3 and self.dir:
                self._reacquire_avatar(grid, diff)
                self._frozen = 0

        # keyboard appears dead (no frame changes at all): the mover may need
        # to be selected first -> try selection actions / clicks briefly
        if self.noop_streak > 20 and self.unlock_left <= 0:
            self.unlock_left = 6
        if self.unlock_left > 0:
            self.unlock_left -= 1
            a, data = self._unlock_action(grid, available)
            self.plan.clear()
            self._last = (a, self.avatar_pos, data)
            return (a, data)

        # nav keeps failing -> the goal may need clicks, not movement
        if self._click_burst > 0:
            self._click_burst -= 1
            a, data = self._click_or_probe(grid, available)
            self._last = (a, self.avatar_pos, data)
            return (a, data)
        if self.no_progress_steps >= 15 and CLICK in available \
                and not self.dir:
            self._click_burst = 8
            self.no_progress_steps = 0

        if self.plan:
            a, data = self._pop_plan_raw()
        elif self.move_actions:
            a, data = self._navigate(grid, available)
        else:
            a, data = self._click_or_probe(grid, available)
        self._last = (a, self.avatar_pos, data)
        return (a, data)

    def _reacquire_avatar(self, grid, diff):
        """The tracked "avatar" is a static decoration — adopt the colors of
        whatever actually changed in the last frame instead."""
        if diff is None:
            return
        mask = diff & ~self._ticking if self._ticking is not None else diff
        if not mask.any() or int(mask.sum()) < 3:
            return
        vals, cnts = np.unique(grid, return_counts=True)
        bg = {int(v) for v, c in zip(vals, cnts) if c > 0.45 * grid.size}
        ys, xs = np.where(mask)
        cols = Counter(int(grid[y, x]) for y, x in zip(ys, xs)
                       if int(grid[y, x]) not in bg)
        new = {c for c, _ in cols.most_common(4)}
        if new and new != self.avatar_colors:
            self.avatar_colors = new
            self.avatar_pos = None
            self.avatar_cells = set()
            self.plan.clear()

    def _unlock_action(self, grid, available):
        """When keys do nothing: cycle selection (ACTION5), undo (ACTION7),
        then click objects to activate a mover."""
        k = self.unlock_left
        if k == 5 and 5 in available:
            return (5, {})
        if k == 4 and 7 in available:
            return (7, {})
        return self._click_or_probe(grid, available)

    def _verify_move(self, new_pos):
        """Check result of last nav action; update blocked/visited/obs."""
        la = self._last
        self._last = None
        if la is None:
            if new_pos is not None:
                self.visited.add(self._lattice(new_pos))
            return
        a, before = la[0], la[1]
        if new_pos is None:
            return
        new_pos = tuple(new_pos)
        if before is not None and a in self.dir:
            if self._lattice(new_pos) == self._lattice(before):
                self._frozen += 1
            else:
                self._frozen = 0
        if before is not None and new_pos != before:
            dx = int(round(new_pos[0] - before[0]))
            dy = int(round(new_pos[1] - before[1]))
            if (dx, dy) != (0, 0) and a in (1, 2, 3, 4):
                # only trust observations along the action's known axis —
                # perpendicular jumps mean the tracked object switched
                if a not in self.dir or \
                        dx * self.dir[a][0] + dy * self.dir[a][1] > 0:
                    self.obs[a].append((dx, dy))
                    self._reclassify()
            self.avatar_pos = new_pos
            self.visited.add(self._lattice(new_pos))
            # self-heal: a cell we actually reached cannot be blocked —
            # failed-move marks record the wall *edge*, not the node
            if self._lattice(new_pos) not in self._traps:
                self.blocked.discard(self._lattice(new_pos))
            sc = self._color_at(new_pos)
            if sc is not None:
                self._safe_colors.add(sc)
            if before is not None and \
                    self._lattice(new_pos) != self._lattice(before):
                self._stuck = 0
        if a not in self.dir or before is None:
            return
        exp = self._predict_dest(before, a)
        if exp and self._lattice(new_pos) != self._lattice(exp):
            ddx = new_pos[0] - before[0]
            ddy = new_pos[1] - before[1]
            ex, ey = self.dir[a]
            dot = ddx * ex + ddy * ey          # forward component
            perp = abs(ddx * ey) + abs(ddy * ex)  # sideways component
            if perp <= self.quantum:
                # stopped short along the expected axis (or didn't move):
                # the unenterable cell is one step past where the avatar
                # actually landed — not the predicted slide destination,
                # which can lie far beyond the true wall.
                ec = self._lattice((new_pos[0] + ex * self.quantum,
                                    new_pos[1] + ey * self.quantum))
                self.blocked.add(ec)
                self._learn_blocker(exp, new_pos)
            # else: moved sideways = selection/tracking noise, ignore
        if self._lattice(new_pos) == self._lattice(before):
            self._stuck += 1
            if self._stuck > 40:
                # immobilized: this cell is a trap — exclude permanently
                self._traps.add(self._lattice(new_pos))
                self.blocked.add(self._lattice(new_pos))
        else:
            self._stuck = 0

    def _predict_dest(self, pos, a):
        """Predicted avatar position after action a from pos."""
        if pos is None or a not in self.dir:
            return pos
        dx, dy = self.dir[a]
        if not self.slide.get(a):
            m = self.mag.get(a, self.quantum)
            return (pos[0] + dx * m, pos[1] + dy * m)
        p = (int(round(pos[0])), int(round(pos[1])))
        for _ in range(64):
            np_ = (p[0] + dx * self.quantum, p[1] + dy * self.quantum)
            if not (0 <= np_[0] < 64 and 0 <= np_[1] < 64):
                break
            if self._edge_blocked(p, np_):
                break
            p = np_
        return p

    def _color_at(self, pos, r=3):
        """Most common non-avatar, non-background color near pos."""
        if self.grid is None or pos is None:
            return None
        h, w = self.grid.shape
        x, y = int(round(pos[0])), int(round(pos[1]))
        vals, counts = np.unique(self.grid, return_counts=True)
        bg = {int(v) for v, c in zip(vals, counts) if c > 0.45 * self.grid.size}
        cnt = {}
        for yy in range(max(0, y - r), min(h, y + r + 1)):
            for xx in range(max(0, x - r), min(w, x + r + 1)):
                c = int(self.grid[yy, xx])
                if c not in self.avatar_colors and c not in bg:
                    cnt[c] = cnt.get(c, 0) + 1
        return max(cnt, key=cnt.get) if cnt else None

    def _record_death(self):
        """The action that ended the game: the color at its destination is a
        hazard candidate. Repeated deaths -> treat the color as blocking and
        never click it again."""
        la = self._last
        self._last = None
        if not la:
            return
        a, before = la[0], la[1]
        data = la[2] if len(la) > 2 else {}
        if a == CLICK and data.get("x") is not None:
            c = self._color_at((data["x"], data["y"]))
            if c is not None:
                self.hazard_count[c] += 1
            # click-order puzzles: after a fatal click, allow re-clicking the
            # surviving objects in a different order
            self.clicked.clear()
            return
        if not self.avatar_colors or a not in self.dir or before is None:
            return
        dest = self._predict_dest(before, a)
        c = self._color_at(dest)
        if c is None or c in self.avatar_colors or c in self._safe_colors:
            return
        self.hazard_count[c] += 1
        if self.hazard_count[c] >= 2:
            self.blocking_colors.add(c)

    def _lattice(self, pos):
        return (int(round(pos[0])), int(round(pos[1])))

    def _learn_blocker(self, exp, before):
        """Mark colors in the *gap* between the current footprint and the
        expected footprint — that's where an obstacle lies. Never marks the
        dominant background color."""
        ex, ey = int(round(exp[0])), int(round(exp[1]))
        bx, by = int(round(before[0])), int(round(before[1]))
        vals, counts = np.unique(self.grid, return_counts=True)
        bg = {int(v) for v, c in zip(vals, counts) if c > 0.45 * self.grid.size}
        w, h = getattr(self, "_av_dims", (1, 1))
        dx, dy = ex - bx, ey - by
        # gap rect strictly between the two footprints along movement axis
        cells = []
        if abs(dx) >= abs(dy):
            s = 1 if dx > 0 else -1
            x0, x1 = bx + s * (w // 2 + 1), ex - s * (w // 2 + 1)
            lo, hi = min(x0, x1), max(x0, x1)
            ylo = min(bx * 0 + by, ey) - h // 2
            yhi = max(by, ey) + h // 2
            for x in range(lo, hi + 1):
                for y in range(ylo, yhi + 1):
                    cells.append((x, y))
        else:
            s = 1 if dy > 0 else -1
            y0, y1 = by + s * (h // 2 + 1), ey - s * (h // 2 + 1)
            lo, hi = min(y0, y1), max(y0, y1)
            xlo = min(bx, ex) - w // 2
            xhi = max(bx, ex) + w // 2
            for y in range(lo, hi + 1):
                for x in range(xlo, xhi + 1):
                    cells.append((x, y))
        if not cells:
            # footprints overlap/adjacent: scan the expected footprint minus old
            cells = [(ex + ddx, ey + ddy)
                     for ddy in range(-(h // 2), h - h // 2)
                     for ddx in range(-(w // 2), w - w // 2)]
        for x2, y2 in cells:
            if 0 <= x2 < self.grid.shape[1] and 0 <= y2 < self.grid.shape[0]:
                if self._ticking is not None and self._ticking[y2, x2]:
                    continue   # animated cells (gauges, counters) aren't walls
                c = int(self.grid[y2, x2])
                if c not in self.avatar_colors and c not in bg:
                    self.blocking_colors.add(c)

    def _pop_plan(self):
        a, data = self._pop_plan_raw()
        return (a, data)

    def _pop_plan_raw(self):
        return self.plan.popleft()

    # ---------- navigation ----------
    def _navigate(self, grid, available):
        if self.avatar_pos is None:
            self._locate_avatar(grid)
            if self.avatar_pos is None:
                return self._random(available)
        start = self._lattice(self.avatar_pos)
        moves = [a for a in self.move_actions if a in available]
        if not moves:
            return self._random(available)

        # precompute wall cells from blocking colors (cheap: direct mask)
        wcells = set()
        if self.blocking_colors:
            ys, xs = np.where(np.isin(grid, list(self.blocking_colors)))
            wcells = set(zip(xs.tolist(), ys.tolist()))
        self._wcells = wcells

        for _ in range(10):
            targets = self._goal_cells(grid)
            if not targets:
                break
            path = self._bfs(start, moves, targets)
            if path is None:
                break
            if path:
                for a in path:
                    self.plan.append((a, {}))
                return self._pop_plan()
            near = min(targets, key=lambda t: abs(t[0]-start[0])+abs(t[1]-start[1]))
            self.bad_targets.add(near)
        # frontier exploration — biased toward candidate-goal objects so the
        # agent explores *toward* likely exits instead of uniformly outward
        path = self._bfs_toward(start, moves, grid)
        if path:
            for a in path:
                self.plan.append((a, {}))
            return self._pop_plan()
        self.no_progress_steps += 1
        return self._random(available)

    def _goal_cells(self, grid):
        """Target cells: centroids of goal-colored objects, else all small
        non-avatar, non-wall objects. Colors with many instances are
        structural (maze lattices, decorations) — never goals."""
        out = set()
        objs = all_objects(grid)
        color_count = Counter(o["color"] for o in objs)
        for o in objs:
            if self.goal_color is not None:
                if o["color"] != self.goal_color:
                    continue
            else:
                if o["color"] in self.avatar_colors or \
                        o["color"] in self.blocking_colors:
                    continue
                if o["size"] > 0.2 * 64 * 64:
                    continue
                if color_count[o["color"]] > 8:
                    continue
                if self._collect_colors and \
                        o["color"] not in self._collect_colors:
                    continue
            c = (int(round(o["centroid"][0])), int(round(o["centroid"][1])))
            if c in self.bad_targets:
                continue
            out.add(c)
        return out

    def _bfs_toward(self, start, moves, grid):
        """Frontier BFS: among all reachable unvisited cells, return the path
        to the one closest to a candidate-goal object (or nearest overall if
        there are none)."""
        seen = {start}
        q = deque([(start, [])])
        found = []          # (pos, path) of reachable unvisited cells
        n = 0
        while q and n < 30000:
            pos, path = q.popleft()
            n += 1
            if pos not in self.visited and pos != start:
                found.append((pos, path))
            for a in moves:
                np_ = self._lattice(self._predict_dest(pos, a))
                if np_ == pos or np_ in seen or np_ in self.blocked:
                    continue
                if not (0 <= np_[0] < 64 and 0 <= np_[1] < 64):
                    continue
                if self._edge_blocked(pos, np_):
                    continue
                seen.add(np_)
                q.append((np_, path + [a]))
        if not found:
            return None
        # reference points: candidate goal objects (even blacklisted ones —
        # they are still informative landmarks)
        refs = [tuple(p) for p in self._goal_locs]
        color_count = Counter(o["color"] for o in all_objects(grid))
        for o in all_objects(grid):
            if o["color"] in self.avatar_colors or \
                    o["color"] in self.blocking_colors:
                continue
            if o["size"] > 0.2 * 64 * 64 or color_count[o["color"]] > 8:
                continue
            refs.append((o["centroid"][0], o["centroid"][1]))
        def score(item):
            pos, path = item
            d_goal = min((abs(pos[0]-r[0])+abs(pos[1]-r[1]) for r in refs),
                         default=0)
            return (d_goal, len(path))
        return min(found, key=score)[1]

    def _bfs(self, start, moves, targets):
        seen = {start}
        q = deque([(start, [])])
        tol = max(2, self.quantum)
        n = 0
        while q and n < 30000:
            pos, path = q.popleft()
            n += 1
            if targets is not None:
                hit = None
                for t in targets:
                    if abs(t[0]-pos[0]) <= tol and abs(t[1]-pos[1]) <= tol:
                        hit = t; break
                if hit is not None:
                    return path
            else:
                if pos not in self.visited and pos != start:
                    return path
            for a in moves:
                np_ = self._lattice(self._predict_dest(pos, a))
                if np_ == pos or np_ in seen or np_ in self.blocked:
                    continue
                if not (0 <= np_[0] < 64 and 0 <= np_[1] < 64):
                    continue
                if self._edge_blocked(pos, np_):
                    continue
                seen.add(np_)
                q.append((np_, path+[a]))
        return None

    def _edge_blocked(self, pos, np_):
        """True if the gap region between avatar footprint at pos and at np_
        contains a learned wall cell (or np_ itself is a wall cell)."""
        wcells = self._wcells
        pos = (int(round(pos[0])), int(round(pos[1])))
        np_ = (int(round(np_[0])), int(round(np_[1])))
        if np_ in wcells or np_ in self.blocked:
            return True
        w, h = getattr(self, "_av_dims", (1, 1))
        dx, dy = np_[0] - pos[0], np_[1] - pos[1]
        if abs(dx) >= abs(dy):
            s = 1 if dx > 0 else -1
            x0, x1 = pos[0] + s * (w // 2 + 1), np_[0] - s * (w // 2 + 1)
            lo, hi = min(x0, x1), max(x0, x1)
            ylo = min(pos[1], np_[1]) - h // 2
            yhi = max(pos[1], np_[1]) + h // 2
            for x in range(lo, hi + 1):
                for y in range(ylo, yhi + 1):
                    if (x, y) in wcells:
                        return True
        else:
            s = 1 if dy > 0 else -1
            y0, y1 = pos[1] + s * (h // 2 + 1), np_[1] - s * (h // 2 + 1)
            lo, hi = min(y0, y1), max(y0, y1)
            xlo = min(pos[0], np_[0]) - w // 2
            xhi = max(pos[0], np_[0]) + w // 2
            for y in range(lo, hi + 1):
                for x in range(xlo, xhi + 1):
                    if (x, y) in wcells:
                        return True
        return False

    def _obj_key(self, prev_grid, pos):
        """(key, cells) of the object containing pos in the previous frame."""
        if prev_grid is None:
            return None, None
        for o in all_objects(prev_grid):
            cells = [tuple(c) for c in o["cells"]]
            if pos in cells:
                key = (o["color"], int(o["centroid"][0]) // 8,
                       int(o["centroid"][1]) // 8)
                return key, cells
        return None, None

    # ---------- click / no-avatar ----------
    def _click_or_probe(self, grid, available):
        if CLICK in available:
            # 1) dwell on "effective" objects (clicks that produced big
            #    changes), alternating with the sweep so new effective
            #    objects keep being discovered. Dwell = press the SAME
            #    object repeatedly — multi-press mechanics reset progress
            #    when a different object is pressed.
            if self._eff_obj and (self._eff_turn or not self._sweep):
                self._eff_turn = False
                for _ in range(len(self._eff_obj)):
                    eo = self._eff_obj[0]
                    if eo["presses"] < 8:
                        eo["presses"] += 1
                        x, y = eo["cells"][eo["presses"] % len(eo["cells"])]
                        self._press_count[(x, y)] += 1
                        return (CLICK, {"x": int(x), "y": int(y)})
                    self._eff_obj.rotate(-1)
                self._eff_obj.clear()
                self._eff_obj_keys.clear()
            else:
                self._eff_turn = True
            # 2) a color whose click already completed a level stays hot
            if self.click_color is not None:
                pri = [o for o in all_objects(grid)
                       if o["color"] == self.click_color]
                if pri:
                    o = self.rng.choice(pri)
                    x, y = self.rng.choice(o["cells"])
                    self._press_count[(x, y)] += 1
                    return (CLICK, {"x": x, "y": y})
            # 3) systematic sweep: salient object cells first (smallest
            #    objects are the most likely targets), then a coarse lattice
            #    covering background/empty positions
            if self._sweep is None:
                self._sweep = deque()
                for o in sorted(all_objects(grid), key=lambda o: o["size"]):
                    if o["size"] > 0.2 * 64 * 64 or \
                            self.hazard_count[o["color"]] >= 2:
                        continue
                    cells = o["cells"]
                    step = max(1, len(cells) // 2)
                    for c in cells[::step][:2]:
                        self._sweep.append(tuple(c))
                for yy in range(0, 64, 4):
                    for xx in range(0, 64, 4):
                        self._sweep.append((xx, yy))
            while self._sweep:
                pos = self._sweep.popleft()
                if pos not in self.clicked:
                    self.clicked.add(pos)
                    x, y = pos
                    self._press_count[pos] += 1
                    return (CLICK, {"x": int(x), "y": int(y)})
            # 4) exhausted: hazard-aware random object clicks
            objs = [o for o in all_objects(grid)
                    if o["size"] <= 0.2 * 64 * 64
                    and self.hazard_count[o["color"]] < 2]
            if objs:
                o = self.rng.choice(objs)
                x, y = self.rng.choice(o["cells"])
                self._press_count[(x, y)] += 1
                return (CLICK, {"x": x, "y": y})
            pos = (self.rng.randint(0, 63), self.rng.randint(0, 63))
            self._press_count[pos] += 1
            return (CLICK, {"x": pos[0], "y": pos[1]})
        return self._random(available)

    def _random(self, available):
        a = self.rng.choice([x for x in available if x != A_RESET])
        data = {}
        if a == CLICK:
            data = {"x": self.rng.randint(0, 63), "y": self.rng.randint(0, 63)}
        return (a, data)


In [ ]:
%%writefile forge_agent.py
"""
ARC-AGI-3 Agent — Forge v5 "Pathfinder" (FIXED)
=================================================
Bug fixes applied:
  [C1] GameAction enum mutation — use from_name() every call for fresh ref
  [C2] ACTION6 coordinate consistency — unified (row=y, col=x) throughout
  [C3] _make_action_input always passes data={}
  [C4] ACTION7 never incremented games_done / corrupted pai index
  [M1] TimeBudget.mark_done() now called on WIN
  [M2] solve_level runs in background thread — choose_action never blocks
  [M3] ACTION7 pai index set to safe value (0)
  [M4] _tensor and _frame_to_tensor merged into single _build_tensor()
  [M5] _inject_expert_demos checks set_level exists before calling
  [M6] Expert injection order fixed (archive before PER reset, inject after)
  [M7] is_done guards lf.state with getattr
  [M8] BFS max_states reduced; game clones not stored in queue
  [M9] _heuristic_score direction reversed (fewer colors = closer to solution)
  [m1] Negative hash in seed fixed with abs() + modulo
  [m2] _NAMES dict includes ACTION7 and full coverage
  [m3] scan_available_actions timeout scaled to budget
  [m4] ActionInput always receives data kwarg
"""

from __future__ import annotations

import copy
import glob
import hashlib
import importlib.util
import logging
import math
import os
import random
import threading
import time
import traceback
from collections import deque, defaultdict
from heapq import heappush, heappop
from typing import Any, Dict, List, Optional, Tuple, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState, ActionInput

logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────

# [m2] Complete action name map including ACTION7
_NAMES: Dict[int, str] = {
    0: 'RESET',
    1: 'ACTION1',
    2: 'ACTION2',
    3: 'ACTION3',
    4: 'ACTION4',
    5: 'ACTION5',
    6: 'ACTION6',
    7: 'ACTION7',
}

# Simple action GameAction list (indices 0-4 map to ACTION1-ACTION5)
_ACTION_MAP = [
    GameAction.ACTION1,
    GameAction.ACTION2,
    GameAction.ACTION3,
    GameAction.ACTION4,
    GameAction.ACTION5,
]

# ─────────────────────────────────────────────────────────────────────────────
# Time Budget Manager
# ─────────────────────────────────────────────────────────────────────────────

class TimeBudget:
    """Hard time-budget: total 8h50m, per-game soft budget."""
    TOTAL_HARD = 8 * 3600 + 50 * 60   # 8h 50min wall-time
    N_GAMES    = 110

    def __init__(self):
        self.start       = time.time()
        self.game_start  = self.start
        self.games_done  = 0
        self._lock       = threading.Lock()

    def new_game(self):
        self.game_start = time.time()

    def mark_done(self):
        # [M1] Called on WIN or game-over so per-game budget stays accurate
        with self._lock:
            self.games_done += 1

    def elapsed(self) -> float:
        return time.time() - self.start

    def remaining_total(self) -> float:
        return max(0.0, self.TOTAL_HARD - self.elapsed())

    def per_game_budget(self) -> float:
        with self._lock:
            remaining_games = max(1, self.N_GAMES - self.games_done)
        return self.remaining_total() / remaining_games

    def game_elapsed(self) -> float:
        return time.time() - self.game_start

    def is_hard_deadline(self) -> bool:
        return self.elapsed() >= self.TOTAL_HARD - 60  # 1-min buffer

    def search_budget(self, fraction: float = 0.50) -> float:
        """Time to give search algorithms (fraction of per-game budget)."""
        return min(self.per_game_budget() * fraction, 180.0)


# Singleton shared across all agent instances
_BUDGET = TimeBudget()


# ─────────────────────────────────────────────────────────────────────────────
# Action helpers
# ─────────────────────────────────────────────────────────────────────────────

def _make_action_input(act_id: int, data: Optional[dict]) -> ActionInput:
    """
    [C3][m4] Always pass data={} so ActionInput never receives None for data.
    For ACTION6 the caller must supply {'x': col, 'y': row}.
    """
    safe_data = data if data is not None else {}
    if act_id == 6:
        return ActionInput(id=GameAction.ACTION6, data=safe_data)
    name = _NAMES.get(act_id, 'ACTION1')
    # [C1] Use from_name to get a fresh reference — never cache enum members
    return ActionInput(id=GameAction.from_name(name), data=safe_data)


def _fresh_action(name: str) -> GameAction:
    """
    [C1] Always return a fresh GameAction reference via from_name so that
    setting .reasoning on it does not mutate a shared enum member.
    """
    return GameAction.from_name(name)


# ─────────────────────────────────────────────────────────────────────────────
# SumTree & Prioritised Experience Replay
# ─────────────────────────────────────────────────────────────────────────────

class SumTree:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.tree     = np.zeros(2 * capacity, dtype=np.float64)
        self.data: List[Optional[dict]] = [None] * capacity
        self.ptr      = 0
        self.size     = 0

    def _propagate(self, idx: int, delta: float):
        while idx > 0:
            idx = (idx - 1) >> 1
            self.tree[idx] += delta

    def update(self, idx: int, priority: float):
        delta = priority - self.tree[idx]
        self.tree[idx] = priority
        self._propagate(idx, delta)

    def add(self, priority: float, data: dict):
        leaf = self.ptr + self.capacity - 1
        self.data[self.ptr] = data
        self.update(leaf, priority)
        self.ptr  = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def get(self, s: float) -> Tuple[int, float, Optional[dict]]:
        idx = 0
        while idx < self.capacity - 1:
            left, right = 2 * idx + 1, 2 * idx + 2
            if s <= self.tree[left]:
                idx = left
            else:
                s  -= self.tree[left]
                idx = right
        data_idx = idx - (self.capacity - 1)
        return idx, self.tree[idx], self.data[data_idx]

    @property
    def total(self) -> float:
        return float(self.tree[0])

    @property
    def min_priority(self) -> float:
        if self.size == 0:
            return 1.0
        leaves = self.tree[self.capacity - 1: self.capacity - 1 + self.size]
        pos    = leaves[leaves > 0]
        return float(pos.min()) if len(pos) > 0 else 1.0

    def __len__(self):
        return self.size


class PERBuffer:
    def __init__(self, capacity=40_000, alpha=0.6, beta_start=0.4,
                 beta_end=1.0, beta_steps=40_000):
        self.tree      = SumTree(capacity)
        self.alpha     = alpha
        self.beta      = beta_start
        self.beta_end  = beta_end
        self.beta_inc  = (beta_end - beta_start) / beta_steps
        self.max_p     = 1.0
        self.seen: set = set()
        self.eps       = 1e-6

    def _key(self, exp: dict) -> str:
        return hashlib.md5(
            exp['s'].tobytes() + str(exp['a']).encode()
        ).hexdigest()[:16]

    def add(self, exp: dict, priority: Optional[float] = None):
        key = self._key(exp)
        if key in self.seen:
            return
        self.seen.add(key)
        p = ((priority if priority is not None else self.max_p) + self.eps) ** self.alpha
        self.tree.add(p, exp)

    def add_expert(self, exp: dict):
        self.add(exp, priority=self.max_p * 5.0)

    def sample(self, batch_size: int):
        if len(self.tree) < batch_size:
            return None, None, None
        batch, idxs, weights = [], [], []
        seg   = self.tree.total / batch_size
        min_p = self.tree.min_priority + self.eps
        self.beta = min(self.beta_end, self.beta + self.beta_inc)
        for i in range(batch_size):
            s            = random.uniform(seg * i, seg * (i + 1))
            idx, p, data = self.tree.get(s)
            if data is None:
                continue
            w     = ((len(self.tree) * (p + self.eps)) ** (-self.beta))
            w_max = ((len(self.tree) * min_p) ** (-self.beta))
            weights.append(w / (w_max + self.eps))
            idxs.append(idx)
            batch.append(data)
        if not batch:
            return None, None, None
        return batch, idxs, np.array(weights, dtype=np.float32)

    def update_priorities(self, idxs: list, td_errors: np.ndarray):
        for idx, err in zip(idxs, td_errors):
            p = (abs(float(err)) + self.eps) ** self.alpha
            self.max_p = max(self.max_p, abs(float(err)) + self.eps)
            self.tree.update(idx, p)

    def __len__(self):
        return len(self.tree)


# ─────────────────────────────────────────────────────────────────────────────
# Cross-level Associative Episodic Memory (CLEAR)
# ─────────────────────────────────────────────────────────────────────────────

class PersistentAEM:
    def __init__(self, capacity=512):
        self.diffs   = deque(maxlen=capacity)
        self.actions = deque(maxlen=capacity)
        self.rewards = deque(maxlen=capacity)

    def push(self, diff: np.ndarray, action: int, reward: float):
        self.diffs.append(diff)
        self.actions.append(action)
        self.rewards.append(reward)

    def get_tensors(self, device, max_mem: int = 64):
        n = min(len(self.diffs), max_mem)
        if n < 2:
            return None, None, None
        diffs = torch.zeros(1, n, 1, 64, 64, device=device)
        acts  = torch.zeros(1, n, dtype=torch.long, device=device)
        rews  = torch.zeros(1, n, device=device)
        for i, (d, a, r) in enumerate(
                list(zip(self.diffs, self.actions, self.rewards))[-n:]):
            diffs[0, i, 0] = torch.from_numpy(d.astype(np.float32))
            acts[0, i]     = min(a, 4)
            rews[0, i]     = r
        return diffs, acts, rews


# ─────────────────────────────────────────────────────────────────────────────
# Neural Network (ForgeNet)
# ─────────────────────────────────────────────────────────────────────────────

class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid      = max(ch // r, 4)
        self.fc1 = nn.Linear(ch, mid)
        self.fc2 = nn.Linear(mid, ch)
        self.sp  = nn.Conv2d(2, 1, 7, padding=3)

    def forward(self, x):
        B, C, H, W = x.shape
        w = torch.sigmoid(self.fc2(F.relu(self.fc1(x.mean(dim=[2, 3])))))
        x = x * w.view(B, C, 1, 1)
        a = torch.sigmoid(self.sp(torch.cat(
            [x.max(1, keepdim=True)[0], x.mean(1, keepdim=True)], 1)))
        return x * a


class AEA(nn.Module):
    def __init__(self, feat_dim=64, mem_dim=32, n_actions=5):
        super().__init__()
        self.mem_dim  = mem_dim
        self.diff_enc = nn.Sequential(
            nn.Conv2d(1, 8, 8, stride=8), nn.ReLU(),
            nn.Conv2d(8, 16, 4, stride=4), nn.ReLU(),
            nn.Flatten(), nn.Linear(16 * 2 * 2, mem_dim))
        self.q_proj = nn.Linear(feat_dim, mem_dim)
        self.v_proj = nn.Linear(mem_dim + 1 + n_actions, n_actions)
        self.scale  = mem_dim ** 0.5

    def forward(self, feat, diffs, actions, rewards):
        B, M = actions.shape
        if M == 0:
            return torch.zeros(B, 5, device=feat.device)
        keys  = self.diff_enc(diffs.reshape(B * M, 1, 64, 64)).reshape(B, M, self.mem_dim)
        q     = self.q_proj(feat).unsqueeze(1)
        attn  = F.softmax(torch.bmm(q, keys.transpose(1, 2)) / self.scale, dim=-1)
        ao    = F.one_hot(actions.clamp(0, 4), 5).float()
        vals  = torch.cat([keys, rewards.unsqueeze(-1), ao], dim=-1)
        ctx   = torch.bmm(attn, vals).squeeze(1)
        return self.v_proj(ctx)


class ForgeNet(nn.Module):
    """Input: 26ch  |  Output: 5 action logits + 64*64 click logits = 4101."""
    def __init__(self, in_ch=26, g=64):
        super().__init__()
        self.g    = g
        self.c1   = nn.Conv2d(in_ch, 32,  3, padding=1)
        self.c2   = nn.Conv2d(32,    64,  3, padding=1)
        self.c3   = nn.Conv2d(64,    128, 3, padding=1)
        self.c4   = nn.Conv2d(128,   256, 3, padding=1)
        self.attn = CBAM(256)
        self.ar   = nn.Conv2d(256, 64, 1)
        self.ap   = nn.MaxPool2d(4, 4)
        self.af   = nn.Linear(64 * 16 * 16, 256)
        self.ah   = nn.Linear(256, 5)
        self.dr   = nn.Dropout(0.15)
        self.cc1  = nn.Conv2d(256, 128, 3, padding=1)
        self.cc2  = nn.Conv2d(128,  64, 3, padding=1)
        self.cc3  = nn.Conv2d(64,   32, 1)
        self.cc4  = nn.Conv2d(32,    1, 1)
        self.gp   = nn.AdaptiveAvgPool2d(1)
        self.gf   = nn.Linear(256, 64)
        self.aea  = AEA(feat_dim=64, mem_dim=32, n_actions=5)

    def forward(self, x, mem_diffs=None, mem_actions=None, mem_rewards=None):
        x  = F.relu(self.c1(x))
        x  = F.relu(self.c2(x))
        x  = F.relu(self.c3(x))
        f  = F.relu(self.c4(x))
        f  = self.attn(f)
        af = F.relu(self.ar(f))
        af = self.ap(af).reshape(f.size(0), -1)
        al = self.ah(self.dr(F.relu(self.af(af))))
        cf = F.relu(self.cc1(f))
        cf = F.relu(self.cc2(cf))
        cf = F.relu(self.cc3(cf))
        cl = self.cc4(cf).reshape(f.size(0), -1)   # (B, 4096)
        if mem_diffs is not None and mem_actions is not None:
            gf = self.gf(self.gp(f).reshape(f.size(0), -1))
            al = al + self.aea(gf, mem_diffs, mem_actions, mem_rewards)
        return torch.cat([al, cl], 1)   # (B, 4101)


# ─────────────────────────────────────────────────────────────────────────────
# Frame / object utilities
# ─────────────────────────────────────────────────────────────────────────────

def _frame_hash(frame: np.ndarray, extras: Optional[dict] = None) -> str:
    h = hashlib.md5(frame.tobytes()).hexdigest()[:16]
    if extras:
        h += "|" + "|".join(f"{k}={v}" for k, v in sorted(extras.items()))
    return h


def _bg_color(frame: np.ndarray) -> int:
    return int(np.bincount(frame.flatten(), minlength=16).argmax())


def _get_objects(frame: np.ndarray, bg: int) -> List[dict]:
    objs = []
    for c in range(16):
        if c == bg:
            continue
        m = frame == c
        n = int(m.sum())
        if 2 <= n <= 3000:
            ys, xs = np.where(m)
            objs.append({
                'c': c,
                'cx': float(xs.mean()), 'cy': float(ys.mean()),
                'n': n, 'xs': xs.tolist(), 'ys': ys.tolist(),
            })
    return objs


def _heuristic_score(frame: np.ndarray, bg: int,
                     initial_frame: Optional[np.ndarray] = None) -> float:
    """
    [M9] Fixed direction: reward states that have changed from the initial
    frame (progress) rather than states with more colors (which perversely
    drove search away from solutions that merge/reduce colors).
    """
    cnt = np.bincount(frame.flatten(), minlength=16)
    cov = 1.0 - cnt[bg] / max(frame.size, 1)
    if initial_frame is not None:
        change = float(np.mean(frame != initial_frame))
        return change * 0.7 + cov * 0.3
    return cov


# ─────────────────────────────────────────────────────────────────────────────
# Action Scanner
# ─────────────────────────────────────────────────────────────────────────────

def scan_available_actions(
        game, f0: np.ndarray, bg: int,
        timeout: float = 6.0) -> List[Tuple[int, Optional[dict]]]:
    """
    Discover which (act_id, data) pairs actually change the frame.
    For ACTION6 probes object centroids + a spatial grid.
    [m3] Timeout increased to 6s default; caller may scale further.
    """
    avail   = getattr(game, '_available_actions', list(range(1, 8)))
    actions: List[Tuple[int, Optional[dict]]] = []

    # Simple actions 1-5, 7
    for a in sorted(set(avail) & {1, 2, 3, 4, 5, 7}):
        g = copy.deepcopy(game)
        try:
            r = g.perform_action(_make_action_input(a, None), raw=True)
            if r.frame and np.any(f0 != np.array(r.frame[-1])):
                actions.append((a, None))
        except Exception:
            pass

    # ACTION6 click probing
    if 6 in avail:
        t0   = time.time()
        seen: Set[str] = set()

        # Build candidate click points
        non_bg = np.argwhere(f0 != bg)
        cands: List[Tuple[int, int]] = []

        # Object centroids + bounding corners
        for obj in _get_objects(f0, bg):
            cands.append((int(round(obj['cx'])), int(round(obj['cy']))))
            xs, ys = obj['xs'], obj['ys']
            for cx, cy in [(min(xs), min(ys)), (max(xs), max(ys)),
                           (min(xs), max(ys)), (max(xs), min(ys))]:
                cands.append((cx, cy))

        # Dense non-bg sample
        if len(non_bg) > 0:
            step = max(1, len(non_bg) // 300)
            for pt in non_bg[::step]:
                cands.append((int(pt[1]), int(pt[0])))

        # Coarse grid backstop
        for y in range(0, 64, 6):
            for x in range(0, 64, 6):
                cands.append((x, y))

        for x, y in cands:
            if time.time() - t0 > timeout:
                break
            x, y = int(np.clip(x, 0, 63)), int(np.clip(y, 0, 63))
            g = copy.deepcopy(game)
            try:
                # [C2] data uses {'x': col, 'y': row} consistently
                data = {'x': x, 'y': y}
                r    = g.perform_action(_make_action_input(6, data), raw=True)
                if r.frame:
                    fh = hashlib.md5(np.array(r.frame[-1]).tobytes()).hexdigest()[:12]
                    if fh not in seen and np.any(f0 != np.array(r.frame[-1])):
                        seen.add(fh)
                        actions.append((6, {'x': x, 'y': y}))
            except Exception:
                pass

    return actions


# ─────────────────────────────────────────────────────────────────────────────
# Hidden state probing utilities
# ─────────────────────────────────────────────────────────────────────────────

def _read_hidden(game, fields: list) -> Optional[dict]:
    if not fields:
        return None
    return {f: getattr(game, f, None) for f in fields
            if getattr(game, f, None) is not None}


def _probe_hidden_fields(game, actions: List[Tuple[int, Optional[dict]]]) -> list:
    init     = {k: v for k, v in game.__dict__.items()
                if isinstance(v, (int, float, bool)) and not k.startswith('__')}
    changing = set()
    for act_id, data in actions[:8]:
        g = copy.deepcopy(game)
        try:
            g.perform_action(_make_action_input(act_id, data), raw=True)
        except Exception:
            continue
        for k, v in g.__dict__.items():
            if (k in init and isinstance(v, (int, float, bool)) and v != init[k]
                    and k not in ('_action_count', '_full_reset', '_action_complete')):
                changing.add(k)
    return sorted(k for k in changing
                  if not k.startswith('_') or k in ('_current_level_index', '_score'))


# ─────────────────────────────────────────────────────────────────────────────
# Beam Search
# ─────────────────────────────────────────────────────────────────────────────

def beam_search(game_factory, level_idx: int,
                actions: List[Tuple[int, Optional[dict]]],
                deadline: float,
                beam_width: int = 64,
                max_depth: int = 40,
                initial_frame: Optional[np.ndarray] = None
                ) -> Optional[List[Tuple[int, Optional[dict]]]]:
    try:
        g0 = game_factory()
        g0.set_level(level_idx)
        g0.perform_action(_make_action_input(0, None), raw=True)
        r0 = g0.perform_action(_make_action_input(0, None), raw=True)
        if not r0.frame:
            return None
        f0  = np.array(r0.frame[-1])
        bg  = _bg_color(f0)
        h0  = _frame_hash(f0)
        lvl = getattr(r0, 'levels_completed', 0) or 0
        i_f = initial_frame if initial_frame is not None else f0.copy()

        beam = [(-_heuristic_score(f0, bg, i_f), copy.deepcopy(g0), [], h0)]
        visited: Set[str] = {h0}

        for depth in range(max_depth):
            if time.time() > deadline:
                break
            candidates = []
            for _, gs, hist, _ in beam:
                for act_id, data in actions:
                    g2 = copy.deepcopy(gs)
                    try:
                        r = g2.perform_action(_make_action_input(act_id, data), raw=True)
                    except Exception:
                        continue
                    if not r.frame:
                        continue
                    f2 = np.array(r.frame[-1])
                    hk = _frame_hash(f2)
                    if hk in visited:
                        continue
                    visited.add(hk)
                    nh  = hist + [(act_id, data)]
                    nlv = getattr(r, 'levels_completed', 0) or 0
                    if nlv > lvl or getattr(g2, '_current_level_index', level_idx) > level_idx:
                        logger.info(f"Beam L{level_idx}: {len(nh)} steps depth={depth+1}")
                        return nh
                    sc = -_heuristic_score(f2, bg, i_f)
                    candidates.append((sc, g2, nh, hk))

            if not candidates:
                break
            candidates.sort(key=lambda x: x[0])
            beam = candidates[:beam_width]

        return None
    except Exception as e:
        logger.warning(f"Beam search error L{level_idx}: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# IDA* Search
# ─────────────────────────────────────────────────────────────────────────────

def ida_star(game_factory, level_idx: int,
             actions: List[Tuple[int, Optional[dict]]],
             deadline: float, max_depth: int = 30,
             initial_frame: Optional[np.ndarray] = None
             ) -> Optional[List[Tuple[int, Optional[dict]]]]:
    try:
        g0 = game_factory()
        g0.set_level(level_idx)
        g0.perform_action(_make_action_input(0, None), raw=True)
        r0 = g0.perform_action(_make_action_input(0, None), raw=True)
        if not r0.frame:
            return None
        f0  = np.array(r0.frame[-1])
        bg  = _bg_color(f0)
        lvl = getattr(r0, 'levels_completed', 0) or 0
        i_f = initial_frame if initial_frame is not None else f0.copy()

        def h(frame):
            # [M9] Lower score = farther from goal; invert for IDA* cost
            return max(0.0, 1.0 - _heuristic_score(frame, bg, i_f))

        result    = [None]
        threshold = h(f0)

        def dfs(g, frame, hist, g_cost, threshold, visited: set):
            if time.time() > deadline:
                return float('inf')
            f_cost = g_cost + h(frame)
            if f_cost > threshold:
                return f_cost
            if len(hist) > max_depth:
                return float('inf')
            min_t = float('inf')
            for act_id, data in actions:
                g2 = copy.deepcopy(g)
                try:
                    r = g2.perform_action(_make_action_input(act_id, data), raw=True)
                except Exception:
                    continue
                if not r.frame:
                    continue
                f2   = np.array(r.frame[-1])
                hk2  = _frame_hash(f2)
                if hk2 in visited:
                    continue
                nlv  = getattr(r, 'levels_completed', 0) or 0
                nh   = hist + [(act_id, data)]
                if nlv > lvl or getattr(g2, '_current_level_index', level_idx) > level_idx:
                    result[0] = nh
                    return -1
                visited.add(hk2)
                t = dfs(g2, f2, nh, g_cost + 1, threshold, visited)
                visited.discard(hk2)
                if t == -1:
                    return -1
                min_t = min(min_t, t)
            return min_t

        while threshold <= max_depth and time.time() < deadline:
            visited = {_frame_hash(f0)}
            t = dfs(copy.deepcopy(g0), f0, [], 0, threshold, visited)
            if t == -1:
                logger.info(f"IDA* L{level_idx}: {len(result[0])} steps")
                return result[0]
            if t == float('inf'):
                break
            threshold = t

        return None
    except Exception as e:
        logger.warning(f"IDA* error L{level_idx}: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# Bounded BFS
# [M8] No longer stores game clones in the queue.
#      Instead stores (frame_hash, history) and reconstructs via replay.
#      max_states reduced from 80k to 15k to stay within memory.
# ─────────────────────────────────────────────────────────────────────────────

def bounded_bfs(game_factory, level_idx: int,
                actions: List[Tuple[int, Optional[dict]]],
                deadline: float,
                max_depth: int = 20,
                max_states: int = 15_000,
                initial_frame: Optional[np.ndarray] = None
                ) -> Optional[List[Tuple[int, Optional[dict]]]]:
    try:
        g0 = game_factory()
        g0.set_level(level_idx)
        g0.perform_action(_make_action_input(0, None), raw=True)
        r0 = g0.perform_action(_make_action_input(0, None), raw=True)
        if not r0.frame:
            return None
        f0  = np.array(r0.frame[-1])
        bg  = _bg_color(f0)
        lvl = getattr(r0, 'levels_completed', 0) or 0
        i_f = initial_frame if initial_frame is not None else f0.copy()
        hidden_fields = _probe_hidden_fields(g0, actions[:6])

        def replay(hist):
            """Reconstruct game state by replaying actions from root."""
            g = game_factory()
            g.set_level(level_idx)
            g.perform_action(_make_action_input(0, None), raw=True)
            g.perform_action(_make_action_input(0, None), raw=True)
            for aid, d in hist:
                r = g.perform_action(_make_action_input(aid, d), raw=True)
            return g, np.array(r.frame[-1]) if r.frame else f0

        h0 = _frame_hash(f0, _read_hidden(g0, hidden_fields))
        # Queue stores (history, depth) — no game clones
        queue:   deque             = deque()
        visited: Dict[str, List]   = {h0: []}
        queue.append(([], 0))
        explored = 0

        while queue and time.time() < deadline and explored < max_states:
            hist, depth = queue.popleft()
            if depth >= max_depth:
                continue

            # Reconstruct state for this history
            try:
                g_cur, f_cur = replay(hist)
            except Exception:
                continue

            for act_id, data in actions:
                g2 = copy.deepcopy(g_cur)
                try:
                    r = g2.perform_action(_make_action_input(act_id, data), raw=True)
                except Exception:
                    continue
                explored += 1
                if not r.frame:
                    continue
                f2  = np.array(r.frame[-1])
                hk  = _frame_hash(f2, _read_hidden(g2, hidden_fields))
                if hk in visited:
                    continue
                nh  = hist + [(act_id, data)]
                nlv = getattr(r, 'levels_completed', 0) or 0
                if nlv > lvl or getattr(g2, '_current_level_index', level_idx) > level_idx:
                    logger.info(f"BFS L{level_idx}: {len(nh)} steps, {explored} exp")
                    return nh
                visited[hk] = nh
                queue.append((nh, depth + 1))

        return None
    except Exception as e:
        logger.warning(f"BFS error L{level_idx}: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# A* Search
# ─────────────────────────────────────────────────────────────────────────────

def astar_search(game_factory, level_idx: int,
                 actions: List[Tuple[int, Optional[dict]]],
                 deadline: float,
                 max_states: int = 50_000,
                 initial_frame: Optional[np.ndarray] = None
                 ) -> Optional[List]:
    try:
        g0 = game_factory()
        g0.set_level(level_idx)
        g0.perform_action(_make_action_input(0, None), raw=True)
        r0 = g0.perform_action(_make_action_input(0, None), raw=True)
        if not r0.frame:
            return None
        f0  = np.array(r0.frame[-1])
        bg  = _bg_color(f0)
        lvl = getattr(r0, 'levels_completed', 0) or 0
        i_f = initial_frame if initial_frame is not None else f0.copy()
        hidden_fields = _probe_hidden_fields(g0, actions[:6])

        def h(frame):
            return 1.0 - _heuristic_score(frame, bg, i_f)

        h0   = _frame_hash(f0, _read_hidden(g0, hidden_fields))
        best: Dict[str, float] = {h0: 0.0}
        heap = []
        ctr  = 0
        heappush(heap, (h(f0), 0, ctr, copy.deepcopy(g0), [], 0))
        explored = 0

        while heap and time.time() < deadline and explored < max_states:
            _, _, _, gs, hist, depth = heappop(heap)
            explored += 1
            if depth > 35:
                continue
            for act_id, data in actions:
                g2 = copy.deepcopy(gs)
                try:
                    r = g2.perform_action(_make_action_input(act_id, data), raw=True)
                except Exception:
                    continue
                if not r.frame:
                    continue
                f2  = np.array(r.frame[-1])
                gc  = depth + 1
                hk  = _frame_hash(f2, _read_hidden(g2, hidden_fields))
                if hk in best and best[hk] <= gc:
                    continue
                best[hk] = gc
                nh  = hist + [(act_id, data)]
                nlv = getattr(r, 'levels_completed', 0) or 0
                if nlv > lvl or getattr(g2, '_current_level_index', level_idx) > level_idx:
                    logger.info(f"A* L{level_idx}: {len(nh)} steps, {explored} exp")
                    return nh
                ctr += 1
                heappush(heap, (gc + h(f2), gc, ctr, g2, nh, gc))

        return None
    except Exception as e:
        logger.warning(f"A* error L{level_idx}: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# MCTS
# ─────────────────────────────────────────────────────────────────────────────

class MCTSNode:
    __slots__ = ('parent', 'action', 'children', 'visits', 'value',
                 'game', 'frame', 'hist', 'is_terminal')
    def __init__(self, parent, action, game, frame, hist):
        self.parent      = parent
        self.action      = action
        self.children: List['MCTSNode'] = []
        self.visits: int  = 0
        self.value: float = 0.0
        self.game         = game
        self.frame        = frame
        self.hist         = hist
        self.is_terminal  = False


def mcts_search(game_factory, level_idx: int,
                actions: List[Tuple[int, Optional[dict]]],
                deadline: float,
                n_rollout: int = 6,
                initial_frame: Optional[np.ndarray] = None
                ) -> Optional[List]:
    try:
        g0 = game_factory()
        g0.set_level(level_idx)
        g0.perform_action(_make_action_input(0, None), raw=True)
        r0 = g0.perform_action(_make_action_input(0, None), raw=True)
        if not r0.frame:
            return None
        f0  = np.array(r0.frame[-1])
        bg  = _bg_color(f0)
        lvl = getattr(r0, 'levels_completed', 0) or 0
        i_f = initial_frame if initial_frame is not None else f0.copy()

        root    = MCTSNode(None, None, copy.deepcopy(g0), f0, [])
        best_r: Optional[List] = None
        C       = math.sqrt(2)

        def ucb1(node: MCTSNode, parent_visits: int) -> float:
            if node.visits == 0:
                return float('inf')
            return (node.value / node.visits +
                    C * math.sqrt(math.log(parent_visits + 1) / node.visits))

        def expand(node: MCTSNode):
            for act_id, data in actions:
                g2 = copy.deepcopy(node.game)
                try:
                    r = g2.perform_action(_make_action_input(act_id, data), raw=True)
                except Exception:
                    continue
                if not r.frame:
                    continue
                f2    = np.array(r.frame[-1])
                nh    = node.hist + [(act_id, data)]
                nlv   = getattr(r, 'levels_completed', 0) or 0
                child = MCTSNode(node, (act_id, data), g2, f2, nh)
                if nlv > lvl or getattr(g2, '_current_level_index', level_idx) > level_idx:
                    child.is_terminal = True
                    return child
                node.children.append(child)
            return None

        def rollout(node: MCTSNode) -> float:
            g, hist = copy.deepcopy(node.game), node.hist[:]
            f       = node.frame.copy()
            for _ in range(n_rollout):
                act_id, data = random.choice(actions)
                try:
                    r = g.perform_action(_make_action_input(act_id, data), raw=True)
                    if not r.frame:
                        continue
                    f   = np.array(r.frame[-1])
                    nlv = getattr(r, 'levels_completed', 0) or 0
                    hist.append((act_id, data))
                    if nlv > lvl or getattr(g, '_current_level_index', level_idx) > level_idx:
                        nonlocal best_r
                        if best_r is None or len(hist) < len(best_r):
                            best_r = hist[:]
                        return 10.0
                except Exception:
                    pass
            return _heuristic_score(f, bg, i_f)

        def backprop(node: MCTSNode, val: float):
            while node is not None:
                node.visits += 1
                node.value  += val
                node = node.parent

        iterations = 0
        while time.time() < deadline:
            node = root
            while node.children and not node.is_terminal:
                node = max(node.children, key=lambda c: ucb1(c, node.visits))
            if not node.is_terminal and node.visits > 0 and len(node.hist) < 30:
                win_child = expand(node)
                if win_child and win_child.is_terminal:
                    backprop(win_child, 10.0)
                    logger.info(f"MCTS L{level_idx}: {len(win_child.hist)} steps, {iterations} iter")
                    return win_child.hist
                if node.children:
                    node = random.choice(node.children)
            val = rollout(node)
            backprop(node, val)
            iterations += 1

        if best_r:
            logger.info(f"MCTS rollout L{level_idx}: {len(best_r)} steps")
        return best_r
    except Exception as e:
        logger.warning(f"MCTS error L{level_idx}: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# Dijkstra Transfer (cross-level solution transfer)
# ─────────────────────────────────────────────────────────────────────────────

def dijkstra_transfer(game_factory, level_idx: int,
                      prev_solution: List[Tuple[int, Optional[dict]]],
                      curr_f0: np.ndarray) -> Optional[List]:
    try:
        pg = game_factory()
        if not hasattr(pg, 'set_level'):
            return None
        pg.set_level(level_idx - 1)
        pg.perform_action(_make_action_input(0, None), raw=True)
        rp = pg.perform_action(_make_action_input(0, None), raw=True)
        if not rp.frame:
            return None
        f_prev    = np.array(rp.frame[-1])
        bg        = _bg_color(curr_f0)
        prev_objs = _get_objects(f_prev, _bg_color(f_prev))
        curr_objs = _get_objects(curr_f0, bg)
        if not prev_objs or not curr_objs:
            return None

        dx_list, dy_list = [], []
        for po in prev_objs:
            best, bd = None, float('inf')
            for co in curr_objs:
                if co['c'] == po['c']:
                    size_ratio = abs(co['n'] - po['n']) / max(po['n'], co['n'], 1)
                    if size_ratio < 0.5:
                        d = abs(co['cx'] - po['cx']) + abs(co['cy'] - po['cy'])
                        if d < bd:
                            bd, best = d, co
            if best:
                dx_list.append(best['cx'] - po['cx'])
                dy_list.append(best['cy'] - po['cy'])
        if not dx_list:
            return None
        dx = float(np.median(dx_list))
        dy = float(np.median(dy_list))

        obj_pts = np.array([[o['cx'], o['cy']] for o in curr_objs])

        transferred = []
        for act_id, data in prev_solution:
            if data and 'x' in data:
                nx = float(np.clip(data['x'] + dx, 0, 63))
                ny = float(np.clip(data['y'] + dy, 0, 63))
                if len(obj_pts) > 0:
                    dists = np.abs(obj_pts[:, 0] - nx) + np.abs(obj_pts[:, 1] - ny)
                    bi    = int(np.argmin(dists))
                    if dists[bi] < 10:
                        nx, ny = obj_pts[bi, 0], obj_pts[bi, 1]
                nd = dict(data)
                # [C2] keep x=col, y=row convention
                nd['x'] = int(round(nx))
                nd['y'] = int(round(ny))
                transferred.append((act_id, nd))
            else:
                transferred.append((act_id, data))

        # Validate
        g = game_factory()
        if not hasattr(g, 'set_level'):
            return None
        g.set_level(level_idx)
        g.perform_action(_make_action_input(0, None), raw=True)
        g.perform_action(_make_action_input(0, None), raw=True)
        for i, (act_id, data) in enumerate(transferred):
            try:
                r   = g.perform_action(_make_action_input(act_id, data), raw=True)
                nlv = getattr(r, 'levels_completed', 0) or 0
                if nlv > level_idx or getattr(g, '_current_level_index', level_idx) > level_idx:
                    logger.info(f"Dijkstra transfer L{level_idx}: {i+1} steps")
                    return transferred[:i + 1]
            except Exception:
                break
        return None
    except Exception as e:
        logger.warning(f"Dijkstra transfer error: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# Game source discovery
# ─────────────────────────────────────────────────────────────────────────────

def find_game_source(game_id: str, arc_env=None) -> Tuple[Optional[str], str]:
    import re
    gid      = game_id.split('-')[0]
    cls_name = gid[0].upper() + gid[1:]
    src      = None

    if arc_env and hasattr(arc_env, 'environment_info'):
        ei = arc_env.environment_info
        if hasattr(ei, 'local_dir') and ei.local_dir:
            from pathlib import Path
            ld = Path(ei.local_dir)
            for c in [ld / f"{gid}.py", ld / f"{cls_name.lower()}.py"]:
                if c.exists():
                    src = str(c)
                    txt = c.read_text()[:3000]
                    m   = re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame', txt)
                    if m:
                        cls_name = m.group(1)
                    break

    if not src:
        search_patterns = [
            f"/tmp/**/{gid}.py",
            f"/kaggle/**/{gid}.py",
            f"**/environment_files/**/{gid}.py",
            f"/kaggle/input/**/{gid}.py",
        ]
        for pat in search_patterns:
            hits = glob.glob(pat, recursive=True)
            if hits:
                src = hits[0]
                try:
                    txt = open(src).read()[:3000]
                    m   = re.search(r'class\s+(\w+)\s*\(\s*ARCBaseGame', txt)
                    if m:
                        cls_name = m.group(1)
                except Exception:
                    pass
                break

    return src, cls_name


# ─────────────────────────────────────────────────────────────────────────────
# Multi-level Solver
# ─────────────────────────────────────────────────────────────────────────────

class MultiSolver:
    """
    Cascade: Dijkstra-transfer → Beam → IDA* → BFS → A* → MCTS
    [M2] solve_level is called from a background thread; never blocks the
         choose_action frame loop.
    """

    def __init__(self, game_path: str, game_cls_name: str, budget: TimeBudget):
        self.game_path     = game_path
        self.game_cls_name = game_cls_name
        self.budget        = budget
        self.game_cls      = None
        self.solutions: Dict[int, List] = {}
        self._loaded       = False

    def load(self) -> bool:
        if self._loaded:
            return self.game_cls is not None
        self._loaded = True
        try:
            spec = importlib.util.spec_from_file_location('_arc_game', self.game_path)
            mod  = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            self.game_cls = getattr(mod, self.game_cls_name)
            logger.info(f"Loaded game class {self.game_cls_name} from {self.game_path}")
            return True
        except Exception as e:
            logger.warning(f"Solver load failed: {e}")
            return False

    def _factory(self):
        return self.game_cls()

    def solve_level(self, level_idx: int, total_budget: float = 120.0) -> Optional[List]:
        if not self.game_cls:
            return None

        deadline_all = time.time() + total_budget
        t0           = time.time()

        # Initial state
        try:
            g0 = self._factory()
            if not hasattr(g0, 'set_level'):
                logger.warning("Game has no set_level method — solver disabled")
                return None
            g0.set_level(level_idx)
            g0.perform_action(_make_action_input(0, None), raw=True)
            r0 = g0.perform_action(_make_action_input(0, None), raw=True)
            if not r0.frame:
                return None
            f0 = np.array(r0.frame[-1])
            bg = _bg_color(f0)
        except Exception as e:
            logger.warning(f"Init failed L{level_idx}: {e}")
            return None

        scan_budget = min(8.0, total_budget * 0.08)
        actions     = scan_available_actions(g0, f0, bg, timeout=scan_budget)
        if not actions:
            logger.warning(f"No effective actions found L{level_idx}")
            return None

        logger.info(f"L{level_idx}: {len(actions)} effective actions, budget={total_budget:.1f}s")

        # Dijkstra transfer from previous level
        if level_idx > 0 and level_idx - 1 in self.solutions:
            try:
                res = dijkstra_transfer(self._factory, level_idx,
                                        self.solutions[level_idx - 1], f0)
                if res:
                    self.solutions[level_idx] = res
                    return res
            except Exception:
                pass

        rem = deadline_all - time.time()
        if rem <= 2:
            return None

        # Budget allocation
        b_beam  = rem * 0.30
        b_ida   = rem * 0.20
        b_bfs   = rem * 0.20
        b_astar = rem * 0.20
        b_mcts  = rem * 0.10

        kwargs = dict(initial_frame=f0)

        # Beam search
        res = beam_search(self._factory, level_idx, actions,
                          deadline=time.time() + b_beam,
                          beam_width=min(128, max(32, len(actions) * 4)),
                          max_depth=40, **kwargs)
        if res:
            self.solutions[level_idx] = res
            return res

        if time.time() > deadline_all:
            return None

        # IDA*
        res = ida_star(self._factory, level_idx, actions,
                       deadline=time.time() + b_ida, max_depth=25, **kwargs)
        if res:
            self.solutions[level_idx] = res
            return res

        if time.time() > deadline_all:
            return None

        # Bounded BFS
        res = bounded_bfs(self._factory, level_idx, actions,
                          deadline=time.time() + b_bfs,
                          max_depth=20, max_states=15_000, **kwargs)
        if res:
            self.solutions[level_idx] = res
            return res

        if time.time() > deadline_all:
            return None

        # A*
        res = astar_search(self._factory, level_idx, actions,
                           deadline=time.time() + b_astar,
                           max_states=50_000, **kwargs)
        if res:
            self.solutions[level_idx] = res
            return res

        if time.time() > deadline_all:
            return None

        # MCTS (last resort)
        res = mcts_search(self._factory, level_idx, actions,
                          deadline=time.time() + b_mcts,
                          n_rollout=8, **kwargs)
        if res:
            self.solutions[level_idx] = res
            return res

        return None


# ─────────────────────────────────────────────────────────────────────────────
# Agent
# ─────────────────────────────────────────────────────────────────────────────

class MyAgent(Agent):
    MAX_ACTIONS = float('inf')
    _MAX_FRAMES  = 10

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # [m1] Use abs() to avoid negative hash; clamp to numpy's uint32 range
        seed = abs(int(time.time() * 1e6) + hash(self.game_id)) % (2 ** 32 - 1)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        global _BUDGET
        _BUDGET.new_game()

        self.device = torch.device(
            'cuda' if torch.cuda.is_available()
            else ('mps' if torch.backends.mps.is_available() else 'cpu'))

        self.G  = 64
        self.IN = 26

        # Per-level state
        self.net: Optional[ForgeNet] = None
        self.opt = None
        self.per = PERBuffer(capacity=40_000)
        self.bsz    = 64
        self.tfreq  = 12
        self.cl     = -1        # current level index
        self.la     = 0         # local action counter
        self.fhist: deque = deque(maxlen=6)
        self.pt  = None         # prev tensor
        self.pai = None         # prev action index
        self.pr  = None         # prev raw frame
        self.ph  = None         # prev frame hash
        self._bg = 0
        self._wm = None
        self._wd = False
        self._eps       = 0.20
        self._eps_min   = 0.04
        self._eps_decay = 0.9995
        self._prev_objs = None
        self._visited_hashes: set = set()
        self._undo_avail  = False
        self._ckpt_hash   = None
        self._unproductive = 0
        self._ttt_steps   = 0
        self._ttt_max     = 3
        self._initial_frame: Optional[np.ndarray] = None

        # Cross-level
        self._clear_archive: deque = deque(maxlen=5000)
        self._paem = PersistentAEM(capacity=512)

        # Solver — runs in a background thread per level
        self._solver: Optional[MultiSolver] = None
        self._solver_init  = False
        self._solution: Optional[List]      = None
        self._sol_step     = 0
        self._solver_thread: Optional[threading.Thread] = None
        self._solution_ready = threading.Event()
        self._solution_lock  = threading.Lock()

    # ── Frame accessors ──────────────────────────────────────────────────────

    def _raw(self, fd) -> np.ndarray:
        return np.array(fd.frame, dtype=np.int64)[-1]

    def _lvl(self, fd) -> int:
        return getattr(fd, 'levels_completed', None) or getattr(fd, 'score', 0) or 0

    # ── Unified tensor construction ──────────────────────────────────────────
    # [M4] Single implementation used for both live inference and training.
    #      update_hist=True advances fhist (live path); False is read-only (training).

    def _build_tensor(self, frame: np.ndarray,
                      update_hist: bool = True) -> torch.Tensor:
        oh  = torch.zeros(16, 64, 64, dtype=torch.float32)
        oh.scatter_(0, torch.from_numpy(frame).unsqueeze(0), 1)

        cnt  = np.bincount(frame.flatten(), minlength=16)
        bg   = int(cnt.argmax())
        if update_hist:
            self._bg = bg
        mx   = max(int(cnt.max()), 1)
        bg_m = (frame == bg).astype(np.float32)
        rar  = np.zeros((64, 64), np.float32)
        for c in range(16):
            if cnt[c] > 0:
                rar[frame == c] = 1.0 - cnt[c] / mx

        pad  = np.pad(frame, 1, mode='edge')
        edge = ((frame != pad[:-2, 1:-1]) | (frame != pad[2:, 1:-1]) |
                (frame != pad[1:-1, :-2]) | (frame != pad[1:-1, 2:])
               ).astype(np.float32)

        # [M4] Fixed: rp varies down rows, cp varies across columns
        rp = np.linspace(0, 1, 64, dtype=np.float32)[:, None] * np.ones((1, 64), dtype=np.float32)
        cp = np.ones((64, 1), dtype=np.float32) * np.linspace(0, 1, 64, dtype=np.float32)[None, :]

        aug = torch.from_numpy(np.stack([bg_m, rar, edge, rp, cp]))

        hist_list = list(self.fhist)
        d1 = torch.zeros(3, 64, 64, dtype=torch.float32)
        for i, prev in enumerate(reversed(hist_list)):
            if i >= 3:
                break
            d1[i] = torch.from_numpy((frame != prev).astype(np.float32))

        d2 = torch.zeros(2, 64, 64, dtype=torch.float32)
        if len(hist_list) >= 2:
            d2[0] = torch.from_numpy((hist_list[-1] != hist_list[-2]).astype(np.float32))
        if len(hist_list) >= 4:
            d2[1] = torch.from_numpy((hist_list[-2] != hist_list[-4]).astype(np.float32))

        if update_hist:
            self.fhist.append(frame.copy())

        return torch.cat([oh, aug, d1, d2], 0)   # (26, 64, 64)

    def _tensor(self, fd) -> torch.Tensor:
        """Live path: updates fhist and moves result to device."""
        return self._build_tensor(self._raw(fd), update_hist=True).to(self.device)

    def _frame_to_tensor(self, frame: np.ndarray) -> torch.Tensor:
        """Training path: read-only, stays on CPU for batching."""
        return self._build_tensor(frame, update_hist=False)

    # ── Reward ───────────────────────────────────────────────────────────────

    def _reward(self, prev: np.ndarray, curr: np.ndarray, curr_h: str) -> float:
        interior = np.ones((64, 64), bool)
        interior[:2] = interior[62:] = False
        changed = bool(np.any((prev != curr) & interior))
        r = 0.0
        if curr_h not in self._visited_hashes:
            r += 1.5
        elif changed:
            r += 0.3
        else:
            r -= 0.1
        if changed:
            r += 0.5
        curr_objs = _get_objects(curr, self._bg)
        if self._prev_objs and curr_objs:
            moved = sum(
                1 for co in curr_objs for po in self._prev_objs
                if co['c'] == po['c']
                and 2 < abs(co['cx'] - po['cx']) + abs(co['cy'] - po['cy']) < 20)
            r += 0.3 * min(moved, 3)
        self._prev_objs = curr_objs
        self._visited_hashes.add(curr_h)
        return r

    # ── Action sampling ──────────────────────────────────────────────────────

    def _sample(self, logits: torch.Tensor, avail=None,
                temp: float = 1.0) -> Tuple[int, Optional[Tuple[int, int]]]:
        """
        Returns (aidx, coords) where:
          aidx 0-4  → ACTION1-ACTION5
          aidx 5    → ACTION6, coords = (row, col) i.e. (y, x)
        [C2] Convention: coords tuple is always (row=y, col=x).
        """
        al   = logits[:5].clone()
        cl   = logits[5:5 + 4096].clone()
        has6 = False

        if avail:
            mask = torch.full((5,), float('-inf'), device=al.device)
            for a in avail:
                aid = a.value if hasattr(a, 'value') else int(a)
                if 1 <= aid <= 5:
                    mask[aid - 1] = 0.0
                elif aid == 6:
                    has6 = True
            al = al + mask

        if not has6:
            cl = torch.full_like(cl, float('-inf'))
        if self._wm is not None:
            cl = cl + torch.log(self._wm.to(self.device).clamp(min=0.01))

        combined = torch.cat([al / temp, cl / temp]).clamp(min=-50.0)
        probs    = F.softmax(combined, dim=0).cpu().numpy()
        probs    = np.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)
        s        = probs.sum()
        if s < 1e-8:
            probs = np.ones(len(probs)) / len(probs)
        else:
            probs /= s

        idx = int(np.random.choice(len(probs), p=probs))
        if idx < 5:
            return idx, None
        # [C2] ci//G = row (y), ci%G = col (x)
        ci = idx - 5
        return 5, (ci // self.G, ci % self.G)

    # ── Click mask ───────────────────────────────────────────────────────────

    @staticmethod
    def _detect_template(frame: np.ndarray, bg: int) -> torch.Tensor:
        mask   = torch.ones(4096, dtype=torch.float32)
        non_bg = (frame != bg).astype(np.int32)
        col_a  = non_bg.sum(axis=0)
        row_a  = non_bg.sum(axis=1)
        idx    = np.arange(4096)
        for c in range(20, 44):
            if col_a[c] <= 2 and col_a[:c].sum() > 0 and col_a[c + 1:].sum() > 0:
                mask[idx % 64 <= c] = 0.05
                return mask
        for r in range(20, 44):
            if row_a[r] <= 2 and row_a[:r].sum() > 0 and row_a[r + 1:].sum() > 0:
                mask[idx // 64 <= r] = 0.05
                return mask
        return mask

    # ── Heuristic warm-up ────────────────────────────────────────────────────

    def _heuristic(self, frame: np.ndarray, avail, step: int
                   ) -> Tuple[int, Optional[Tuple[int, int]]]:
        av = {int(a.value) if hasattr(a, 'value') else int(a) for a in avail}
        for d in [1, 2, 3, 4]:
            if d in av and step < 4:
                return d - 1, None
        if 6 in av:
            cnt     = np.bincount(frame.flatten(), minlength=16)
            targets = []
            for c in range(16):
                if c == self._bg or cnt[c] == 0 or cnt[c] > 2000:
                    continue
                ys, xs = np.where(frame == c)
                if len(ys) >= 2:
                    targets.append((int(np.median(xs)), int(np.median(ys)), len(ys)))
            targets.sort(key=lambda t: t[2])
            pidx = step - 4
            if 0 <= pidx < len(targets):
                # col=x, row=y
                return 5, (targets[pidx][1], targets[pidx][0])
        choices = [a for a in av if 1 <= a <= 5]
        if choices:
            return random.choice(choices) - 1, None
        return 0, None

    # ── Training ─────────────────────────────────────────────────────────────

    def _train(self):
        if len(self.per) < self.bsz:
            return
        batch, idxs, weights = self.per.sample(self.bsz)
        if batch is None:
            return
        arch_n = min(self.bsz // 4, len(self._clear_archive))
        if arch_n > 0:
            arch    = random.sample(list(self._clear_archive), arch_n)
            batch   = batch[:self.bsz - arch_n] + arch
            weights = np.concatenate([weights[:self.bsz - arch_n],
                                      np.ones(arch_n, dtype=np.float32)])

        states = torch.stack([self._frame_to_tensor(e['s']).to(self.device)
                               for e in batch])
        acts   = torch.tensor([e['a'] for e in batch],
                               dtype=torch.long, device=self.device)
        rews   = torch.tensor([e['r'] for e in batch],
                               dtype=torch.float32, device=self.device)
        ws     = torch.tensor(weights, device=self.device)
        rews   = torch.sigmoid(rews)

        self.opt.zero_grad()
        logits  = self.net(states)
        acts_c  = acts.clamp(0, logits.size(1) - 1)
        sel     = logits.gather(1, acts_c.unsqueeze(1)).squeeze(1)
        el      = F.binary_cross_entropy_with_logits(sel, rews, reduction='none')
        loss    = (el * ws).mean()
        p       = torch.sigmoid(logits)
        loss    = loss - 0.0001 * p[:, :5].mean() - 0.00001 * p[:, 5:].mean()
        loss.backward()
        nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
        self.opt.step()

        n_per = len(idxs)
        if n_per > 0:
            self.per.update_priorities(idxs[:n_per],
                                       el[:n_per].detach().cpu().numpy())

    # ── TTT (test-time training) ──────────────────────────────────────────────

    def _ttt_step(self, tensor: torch.Tensor):
        if self._ttt_steps >= self._ttt_max or self.opt is None:
            return
        self.net.train()
        self.opt.zero_grad()
        t1 = tensor.unsqueeze(0)
        t2 = torch.roll(t1, shifts=random.randint(0, 3), dims=-1)
        with torch.no_grad():
            l2 = self.net(t2)
        l1   = self.net(t1)
        loss = F.mse_loss(l1, l2.detach()) * 0.01
        loss.backward()
        self.opt.step()
        self.net.eval()
        self._ttt_steps += 1

    # ── Archive ───────────────────────────────────────────────────────────────

    def _archive_level(self):
        """[M6] Archive current PER contents BEFORE resetting PER."""
        if len(self.per) == 0:
            return
        batch, _, _ = self.per.sample(min(200, len(self.per)))
        if batch:
            self._clear_archive.extend(batch)

    # ── Expert injection from solver solution ─────────────────────────────────

    def _inject_expert_demos(self, level_idx: int):
        if not (self._solver and self._solver.game_cls):
            return
        sol = self._solver.solutions.get(level_idx)
        if not sol:
            return
        try:
            g = self._solver.game_cls()
            # [M5] Guard: check set_level exists before calling
            if not hasattr(g, 'set_level'):
                logger.warning("Game has no set_level — skipping expert injection")
                return
            g.set_level(level_idx)
            g.perform_action(_make_action_input(0, None), raw=True)
            g.perform_action(_make_action_input(0, None), raw=True)
            for act_id, data in sol:
                try:
                    pf = np.array(g.get_pixels(0, 0, 64, 64), dtype=np.int64)
                except Exception:
                    pf = self.pr.copy() if self.pr is not None else np.zeros((64, 64), np.int64)
                g.perform_action(_make_action_input(act_id, data), raw=True)
                if act_id <= 5:
                    a_idx = act_id - 1
                elif act_id == 6 and data:
                    # [C2] row=y, col=x
                    a_idx = 5 + data.get('y', 0) * 64 + data.get('x', 0)
                else:
                    a_idx = 0
                exp = {'s': pf.copy(), 'a': a_idx, 'r': 2.0}
                self.per.add_expert(exp)
                self._clear_archive.append(exp)
            if len(self.per) >= self.bsz:
                for _ in range(min(15, len(self.per) // max(self.bsz, 1))):
                    self._train()
        except Exception as e:
            logger.warning(f"Expert inject L{level_idx}: {e}")

    # ── Background solver launcher ────────────────────────────────────────────

    def _launch_solver(self, level_idx: int):
        """[M2] Run solve_level in a daemon thread. choose_action never blocks."""
        if not (self._solver and self._solver.game_cls):
            self._solution_ready.set()
            return

        search_time = min(_BUDGET.search_budget(0.50), 90.0)
        if level_idx > 0:
            search_time = min(search_time * (1 + 0.08 * level_idx), 160.0)

        self._solution_ready.clear()

        def _run():
            try:
                sol = self._solver.solve_level(level_idx, total_budget=search_time)
                with self._solution_lock:
                    if sol:
                        self._solution = sol
            except Exception as e:
                logger.warning(f"Solver thread error L{level_idx}: {e}")
            finally:
                self._solution_ready.set()

        self._solver_thread = threading.Thread(target=_run, daemon=True)
        self._solver_thread.start()

    # ── is_done ───────────────────────────────────────────────────────────────

    def is_done(self, frames, lf) -> bool:
        try:
            # [M7] Guard lf.state access
            state = getattr(lf, 'state', None)
            if state == GameState.WIN:
                _BUDGET.mark_done()   # [M1] increment so per-game budget stays accurate
                return True
            if _BUDGET.is_hard_deadline():
                return True
            # Per-game soft budget: end if we've spent 2.5× per-game budget
            per_game = _BUDGET.per_game_budget()
            if _BUDGET.game_elapsed() > per_game * 2.5:
                logger.info(f"Per-game budget exceeded ({_BUDGET.game_elapsed():.0f}s), "
                            f"per_game={per_game:.0f}s")
                _BUDGET.mark_done()
                return True
            return False
        except Exception:
            return True

    # ── choose_action — must return a GameAction enum member ──────────────────

    def choose_action(self, frames, lf) -> GameAction:
        try:
            lvl = self._lvl(lf)

            # ── Level transition ──────────────────────────────────────────────
            if lvl != self.cl:
                # Lazy solver init
                if not self._solver_init:
                    self._solver_init = True
                    src, cls = find_game_source(
                        self.game_id, getattr(self, 'arc_env', None))
                    if src:
                        self._solver = MultiSolver(src, cls, _BUDGET)
                        self._solver.load()

                # [M6] Archive BEFORE resetting PER
                self._archive_level()

                # Reset per-level state
                with self._solution_lock:
                    self._solution = None
                self._sol_step = 0
                self.per  = PERBuffer(capacity=40_000)
                self.net  = ForgeNet(self.IN, self.G).to(self.device)

                # Load pretrained weights if available
                for wp in [
                    '/kaggle/input/forge-pretrained-weights/pretrained_weights.pt',
                    'pretrained_weights.pt',
                ]:
                    try:
                        if os.path.exists(wp):
                            saved = torch.load(wp, map_location=self.device,
                                               weights_only=True)
                            ms = self.net.state_dict()
                            ms.update({k: v for k, v in saved.items()
                                       if k in ms and v.shape == ms[k].shape})
                            self.net.load_state_dict(ms)
                            logger.info(f"Loaded pretrained weights from {wp}")
                            break
                    except Exception:
                        pass

                self.net.eval()
                self.opt = optim.Adam(self.net.parameters(), lr=3e-4)

                # Reset per-level counters
                self.pt    = self.pai = self.pr = self.ph = None
                self.cl    = lvl
                self.la    = 0
                self.fhist.clear()
                self._wd              = False
                self._wm              = None
                self._eps             = 0.20
                self._prev_objs       = None
                self._visited_hashes  = set()
                self._ckpt_hash       = None
                self._unproductive    = 0
                self._ttt_steps       = 0
                self._initial_frame   = None

                # Inject expert demos from the PREVIOUS level's solution
                if lvl > 0:
                    self._inject_expert_demos(lvl - 1)

                # [M2] Launch solver in background thread — does not block here
                self._launch_solver(lvl)

            # ── Hard deadline guard ───────────────────────────────────────────
            if _BUDGET.is_hard_deadline():
                act = _fresh_action('RESET')
                act.reasoning = {"action": "RESET", "reason": "hard_deadline"}
                return act

            # ── [M7] Safe state access ────────────────────────────────────────
            state = getattr(lf, 'state', None)

            if state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                self.pt = self.pai = self.pr = self.ph = None
                act = _fresh_action('RESET')
                act.reasoning = {"action": "RESET", "reason": "state_reset"}
                return act

            # ── Record initial frame for heuristic baseline ───────────────────
            raw = self._raw(lf)
            if self._initial_frame is None:
                self._initial_frame = raw.copy()

            # ── Solution playback ─────────────────────────────────────────────
            # Check if solver has delivered a solution since last check
            with self._solution_lock:
                sol_available = (self._solution is not None and
                                 self._sol_step < len(self._solution))

            if sol_available:
                with self._solution_lock:
                    act_id, data = self._solution[self._sol_step]
                    self._sol_step += 1
                    sol_total = len(self._solution)

                self.fhist.append(raw.copy())
                self.pr  = raw.copy()
                self.la += 1

                if act_id == 6:
                    # [C1] Fresh ref each call
                    act = _fresh_action('ACTION6')
                    act.reasoning = {
                        "action": "ACTION6",
                        # [C2] data['x']=col, data['y']=row
                        "data": {"x": int(data['x']), "y": int(data['y'])},
                        "reason": f"sol:{self._sol_step}/{sol_total}",
                    }
                else:
                    name = _NAMES.get(act_id, 'ACTION1')
                    act  = _fresh_action(name)
                    act.reasoning = {
                        "action": name,
                        "reason": f"sol:{self._sol_step}/{sol_total}",
                    }
                return act

            # ── Online RL policy ──────────────────────────────────────────────
            tensor = self._tensor(lf)
            ch     = hashlib.md5(raw.tobytes()).hexdigest()[:16]
            avail  = getattr(lf, 'available_actions', None) or []
            self._undo_avail = any(
                (a.value if hasattr(a, 'value') else int(a)) == 7 for a in avail)

            # Store transition
            if self.pt is not None and self.pai is not None:
                r   = self._reward(self.pr, raw, ch)
                exp = {'s': self.pr.copy(), 'a': self.pai, 'r': r}
                self.per.add(exp)
                interior = np.ones((64, 64), bool)
                interior[:2] = interior[62:] = False
                if np.any((self.pr != raw) & interior):
                    self._paem.push((self.pr != raw).astype(bool) & interior,
                                    min(self.pai, 4), r)
                    self._ckpt_hash    = ch
                    self._unproductive = 0
                else:
                    self._unproductive += 1

            if self._wm is None:
                self._wm = self._detect_template(raw, self._bg)

            # Undo if stuck
            if self._undo_avail and self._unproductive >= 25 and self._ckpt_hash:
                self._unproductive = 0
                act = _fresh_action('ACTION7')
                act.reasoning = {"action": "ACTION7", "reason": "undo_stuck"}
                self.pt  = tensor
                # [M3] Use a safe neutral action index (0 = ACTION1) not 6
                self.pai = 0
                self.pr  = raw.copy()
                self.ph  = ch
                self.la += 1
                return act

            # Warm-up vs trained policy
            aidx, coords = 0, None
            if not self._wd:
                if self.la < 10:
                    aidx, coords = self._heuristic(raw, avail, self.la)
                else:
                    self._wd = True
                    for _ in range(min(5, len(self.per) // max(self.bsz, 1))):
                        self._train()

            if self._wd:
                self._ttt_step(tensor)
                if random.random() < self._eps:
                    aidx, coords = self._sample(
                        torch.zeros(4101, device=self.device), avail, temp=2.5)
                else:
                    self.net.eval()
                    with torch.no_grad():
                        mem = self._paem.get_tensors(self.device, max_mem=64)
                        if mem[0] is not None:
                            logits = self.net(tensor.unsqueeze(0), *mem).squeeze(0)
                        else:
                            logits = self.net(tensor.unsqueeze(0)).squeeze(0)
                    aidx, coords = self._sample(logits, avail, temp=0.5)
                self._eps = max(self._eps_min, self._eps * self._eps_decay)
            elif self.la >= 10:
                self._wd = True

            # ── Build the returned GameAction ─────────────────────────────────
            if aidx < 5:
                # [C1] Fresh reference — do not mutate shared enum member
                name = _NAMES[aidx + 1]       # aidx 0 → ACTION1, etc.
                act  = _fresh_action(name)
                act.reasoning  = {"action": name, "reason": f"rl:a{aidx+1}"}
                action_idx     = aidx
            else:
                # aidx == 5 → ACTION6
                # [C2] coords = (row=y, col=x)
                row, col = coords
                act = _fresh_action('ACTION6')
                act.reasoning = {
                    "action": "ACTION6",
                    "data": {"x": int(col), "y": int(row)},
                    "reason": f"rl:c(col={col},row={row})",
                }
                action_idx = 5 + row * self.G + col

            self.pt  = tensor
            self.pai = action_idx
            self.pr  = raw.copy()
            self.ph  = ch
            self.la += 1

            if self.la % self.tfreq == 0 and self._wd:
                self._train()
                if (self.la // self.tfreq) % 5 == 0:
                    self._ttt_steps = 0

            return act

        except Exception as e:
            traceback.print_exc()
            # [C1] Always fresh reference on fallback too
            act = _fresh_action(_ACTION_MAP[random.randint(0, 4)].name)
            act.reasoning = {"action": act.name, "reason": f"err:{str(e)[:80]}"}
            return act

In [ ]:
%%writefile run_local.py
"""Fast local runner for OceanCore against LocalEnvironmentWrapper.

Usage: .venv312/bin/python run_local.py ls20 [max_actions] [seed]
       .venv312/bin/python run_local.py all 1500
"""
import json, sys, logging, importlib
from pathlib import Path
sys.path.insert(0, "/home/takumu/kaggle/arc-agi-3/ARC-AGI-3-Agents")
sys.path.insert(0, "/home/takumu/kaggle/arc-agi-3")
from arc_agi.local_wrapper import LocalEnvironmentWrapper
from arc_agi.models import EnvironmentInfo
from arcengine import GameAction
logging.disable(logging.CRITICAL)
import numpy as np

import agent_core
importlib.reload(agent_core)
from agent_core import OceanCore

BASE = Path("/home/takumu/kaggle/arc-agi-3/environment_files")


def load(game, seed=0):
    sub = next((BASE/game).iterdir())
    meta = json.loads((sub/"metadata.json").read_text())
    info = EnvironmentInfo(**meta); info.local_dir = str(sub)
    env = LocalEnvironmentWrapper(info, logging.getLogger("t"), scorecard_id="x", seed=seed)
    return env, meta


def run_game(game, max_actions=1500, seed=0, verbose=False):
    env, meta = load(game, seed)
    core = OceanCore(seed=seed)
    baselines = meta.get("baseline_actions", [])
    raw = env.step(GameAction.RESET)
    if raw is None or not raw.frame:
        return {"game": game, "error": "reset failed"}
    level_actions = 0
    level_scores = []
    per_level_actions = []
    prev_level = raw.levels_completed
    total_actions = 0
    avail = raw.available_actions
    while total_actions < max_actions:
        grid = np.asarray(raw.frame[-1])
        aid, data = core.decide(grid, raw.available_actions, raw.state.value if hasattr(raw.state,'value') else str(raw.state),
                              raw.levels_completed, total_actions)
        act = GameAction.from_id(aid)
        raw = env.step(act, data if data else None)
        total_actions += 1
        if raw is None or not raw.frame:
            # invalid action or dead-end; keep going
            raw = env.step(GameAction.RESET)
            continue
        if raw.state.name == "NOT_FINISHED":
            level_actions += 1
        if raw.levels_completed > prev_level:
            per_level_actions.append(level_actions)
            b = baselines[prev_level] if prev_level < len(baselines) else 30
            sc = min((b / max(level_actions,1))**2 * 100, 115.0)
            level_scores.append(sc)
            if verbose:
                print(f"  [{game}] level {prev_level} done in {level_actions} (base {b}) -> {sc:.1f}")
            level_actions = 0
            prev_level = raw.levels_completed
        if raw.state.name == "WIN":
            break
    return {"game": game, "levels": prev_level, "win_levels": raw.win_levels,
            "level_scores": level_scores, "per_level_actions": per_level_actions,
            "actions": total_actions, "state": raw.state.name}


def game_score(r, baselines):
    """Weighted-by-index average of level scores (levels not completed = 0)."""
    n = r["win_levels"]
    if n == 0: return 0.0
    scores = r["level_scores"] + [0.0]*(n - len(r["level_scores"]))
    num = sum((i+1)*s for i,s in enumerate(scores[:n]))
    den = sum(range(1, n+1))
    maxw = sum((i+1) for i,s in enumerate(scores[:n]) if s>0)
    return min(num/den, maxw/den*100)


if __name__ == "__main__":
    args = sys.argv[1:]
    games = args[0].split(",") if args else ["ls20"]
    max_actions = int(args[1]) if len(args)>1 else 1500
    verbose = "-v" in args
    if games == ["all"]:
        games = sorted(p.name for p in BASE.iterdir())
    total = 0.0
    for g in games:
        r = run_game(g, max_actions, verbose=verbose)
        if "error" in r:
            print(f"{g:6} ERROR {r['error']}"); continue
        _, meta = load(g)
        gs = game_score(r, meta.get("baseline_actions",[]))
        total += gs
        print(f"{g:6} levels={r['levels']}/{r['win_levels']} actions={r['actions']:5} "
              f"level_actions={r['per_level_actions']} score={gs:.2f} state={r['state']}")
    print(f"\nMEAN game score over {len(games)}: {total/len(games):.2f}%")



## Quick local smoke test

With the `arc-agi` package and the official `ARC-AGI-3-Agents` framework on
`sys.path`, either agent can be driven through the same `Agent.main()` loop the
Kaggle gateway uses:

```python
import arc_agi
from arc_agi import OperationMode
arc = arc_agi.Arcade(operation_mode=OperationMode.NORMAL)
env = arc.make('m0r0')
agent = MyAgent(card_id='local', game_id='m0r0', agent_name='a',
                ROOT_URL='http://localhost', record=False, arc_env=env)
agent.main()
print(agent.frames[-1].levels_completed, agent.action_counter)
```
